# Observatorio NOA — v3.2.0 · SS3 como planificador semántico **único**
**Experimento SS3 → Retrieval → EvidenceDossier → Ministral-3-8B → validación**  

**Contrato arquitectónico:** no hay planner determinístico, Gold literal, sombra, votaciones, fusión ni rutas A/B. La única fuente de `operation`, operaciones secundarias, dimensión, hipótesis y riesgos es PySS3. Las búsquedas literales de provincias, fuentes y fechas son **extracción de alcance**, no clasificadores de intención. Los contratos documentales sólo supervisan evidencia y publicación.

**Propósito:** comprobar una arquitectura de captura semántica por SS3 con autoridad completa, en una prueba replicable. El entrenamiento ampliado es **sintético y provisional**; los resultados de esta notebook, no las expectativas del autor, decidirán si se validó. El conjunto externo de 80 casos no se usa para entrenamiento ni calibración. El LLM necesita GPU y acceso al modelo real. Un fallo se registra, nunca se sustituye con respuestas simuladas.

## 1. Dependencias y configuración reproducible

In [1]:
%pip -q install \
    "pandas>=2.1,<3" "pyarrow>=15" "numpy>=1.26,<3" \
    "scikit-learn>=1.4,<2" "sentence-transformers>=3,<6" \
    "pyss3==0.6.5" "rank-bm25>=0.2,<1" "transformers>=4.57" \
    "accelerate>=1" "mistral-common>=1.8" "gradio>=6,<7" \
    "reportlab>=4.2,<5" "pypdf>=4,<7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.1/262.1 kB 21.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 149.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.4/161.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 30.4 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import gc, hashlib, json, math, os, re, time, unicodedata, warnings, zipfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

VERSION = "3.2.0-SS3-SOLO"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_DIR = Path("/content/observatorio_noa_v320_ss3_solo")
INPUT_DIR = BASE_DIR / "input"
WORK_DIR = BASE_DIR / "work"
OUTPUT_DIR = BASE_DIR / "outputs"
AUDIT_DIR = OUTPUT_DIR / "audit"
for d in [INPUT_DIR, WORK_DIR, OUTPUT_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RETRIEVAL_PACKAGE_NAMES = [
    "observatorio_noa_rag_v302.zip",
    "observatorio_noa_rag_v302(1).zip",
]
CLOSEOUT_PACKAGE_NAMES = ["observatorio_noa_retrieval_v303_closeout.zip"]

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MINISTRAL_MODEL_ID = "mistralai/Ministral-3-8B-Instruct-2512"

MAX_DOCUMENTS = 8
MAX_EVIDENCE_ITEMS = 10
MAX_CHUNKS_PER_DOCUMENT = 2
MAX_CONTEXT_CHARS_PER_EVIDENCE = 1400
MAX_TOTAL_EVIDENCE_CHARS = 16000

ENABLE_RERANKER = False
MIN_SENTENCE_EVIDENCE_COSINE = 0.15
LLM_MAX_NEW_TOKENS = 560

PROVINCES = ["Catamarca", "La Rioja", "Santiago del Estero"]
SOURCES = ["CFI", "SAIJ"]
PERIODS = ["2009–2011", "2012–2015", "2016–2019"]

EXPECTED_CLOSEOUT_STATUS = "CLOSED_READY_FOR_LLM"
EXPECTED_MIN_RECALL_AT_8 = 0.85
EXPECTED_RUNTIME_SIGNATURE = "c34acdb5692df44c35b7e12a9d2c3cbb6061aa37eff24d5157b93c4c44adf86f"

V31_BASELINE = {
    "mean_substantive_claim_acceptance": 0.5253260395067118,
    "total_rejected_units": 42,
    "mean_citation_completeness": 1.0,
    "hypothesis_adversarial_handling": 0.0,
}

SS3_SOLE_SEMANTIC_AUTHORITY = True
PLANNER_IMPLEMENTATION = 'PYSS3_ONLY_V320'
assert SS3_SOLE_SEMANTIC_AUTHORITY


## 2. Corpus e índices congelados · comprobación de custodia

In [3]:
def find_existing(names: list[str]) -> Path | None:
    for root in [Path("/content"), Path("/mnt/data"), INPUT_DIR]:
        for name in names:
            candidate = root / name
            if candidate.exists():
                return candidate
    return None

retrieval_zip = find_existing(RETRIEVAL_PACKAGE_NAMES)
closeout_zip = find_existing(CLOSEOUT_PACKAGE_NAMES)

if retrieval_zip is None or closeout_zip is None:
    try:
        from google.colab import files
        print(
            "Suba dos archivos:\n"
            "1) observatorio_noa_rag_v302.zip\n"
            "2) observatorio_noa_retrieval_v303_closeout.zip"
        )
        uploaded = files.upload()
        for filename, content in uploaded.items():
            (INPUT_DIR / filename).write_bytes(content)
        retrieval_zip = retrieval_zip or find_existing(RETRIEVAL_PACKAGE_NAMES)
        closeout_zip = closeout_zip or find_existing(CLOSEOUT_PACKAGE_NAMES)
    except Exception:
        pass

if retrieval_zip is None:
    raise FileNotFoundError("Falta observatorio_noa_rag_v302.zip")
if closeout_zip is None:
    raise FileNotFoundError(
        "Falta observatorio_noa_retrieval_v303_closeout.zip. "
        "v3.1.1 no ejecuta sin el cierre formal v3.0.3."
    )

RUNTIME_DIR = WORK_DIR / "retrieval_v302"
CLOSEOUT_DIR = WORK_DIR / "closeout_v303"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
CLOSEOUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(retrieval_zip) as archive:
    archive.extractall(RUNTIME_DIR)
with zipfile.ZipFile(closeout_zip) as archive:
    archive.extractall(CLOSEOUT_DIR)

runtime_manifest = json.loads(
    (RUNTIME_DIR / "rag_run_manifest_v302.json").read_text(encoding="utf-8")
)

report_path = CLOSEOUT_DIR / "retrieval_closeout_report_v303.json"
gold_path = CLOSEOUT_DIR / "rag_goldset_reviewed.csv"

if not report_path.exists() or not gold_path.exists():
    raise FileNotFoundError("El paquete de cierre v3.0.3 está incompleto")

closeout_report = json.loads(report_path.read_text(encoding="utf-8"))
gold_df = pd.read_csv(gold_path)

status = closeout_report.get("retrieval_stage_status")
recall = closeout_report.get(
    "recall_at_8_mean",
    closeout_report.get("hard_gold_metric", {}).get("recall_at_8_mean"),
)

if status != EXPECTED_CLOSEOUT_STATUS:
    raise RuntimeError(f"Retrieval no está formalmente cerrado: {status!r}")
if recall is None or float(recall) < EXPECTED_MIN_RECALL_AT_8:
    raise RuntimeError(f"Recall@8 no verificable o insuficiente: {recall}")
if set(gold_df["gold_status"].astype(str)) != {"reviewed"}:
    raise RuntimeError("El Gold Set no está completamente revisado")

print("Retrieval:", retrieval_zip)
print("Cierre:", closeout_zip)
print("Status:", status)
print("Recall@8:", recall)
print("Gold:", len(gold_df), "preguntas")

Suba dos archivos:
1) observatorio_noa_rag_v302.zip
2) observatorio_noa_retrieval_v303_closeout.zip


Saving observatorio_noa_rag_v302.zip to observatorio_noa_rag_v302.zip
Saving observatorio_noa_retrieval_v303_closeout.zip to observatorio_noa_retrieval_v303_closeout.zip
Retrieval: /content/observatorio_noa_rag_v302.zip
Cierre: /content/observatorio_noa_retrieval_v303_closeout.zip
Status: CLOSED_READY_FOR_LLM
Recall@8: 0.9583333333333334
Gold: 7 preguntas


In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

required_runtime_files = [
    "rag_document_index.parquet",
    "rag_retrieval_section_index.parquet",
    "rag_chunk_index.parquet",
    "rag_document_embeddings.npy",
    "rag_chunk_embeddings.npy",
    "rag_retrieval_section_embeddings.npy",
    "rag_document_embedding_map.csv",
    "rag_chunk_embedding_map.csv",
    "rag_lda_k2_descriptors.csv",
    "rag_lda_k3_descriptors.csv",
    "rag_microtopic_descriptors.csv",
    "rag_lda_k2_descriptor_embeddings.npy",
    "rag_lda_k3_descriptor_embeddings.npy",
    "rag_microtopic_descriptor_embeddings.npy",
    "rag_goldset_review_candidates.csv",
]
missing = [f for f in required_runtime_files if not (RUNTIME_DIR / f).exists()]
if missing:
    raise FileNotFoundError("Faltan archivos del runtime: " + ", ".join(missing))

runtime_hashes_df = pd.DataFrame([
    {
        "filename": filename,
        "sha256": sha256_file(RUNTIME_DIR / filename),
        "size_bytes": int((RUNTIME_DIR / filename).stat().st_size),
    }
    for filename in required_runtime_files
])
RUNTIME_SIGNATURE = hashlib.sha256(
    "\n".join(f"{r.filename}:{r.sha256}" for r in runtime_hashes_df.itertuples(index=False)).encode("utf-8")
).hexdigest()
print("Runtime signature:", RUNTIME_SIGNATURE)
display(runtime_hashes_df.head())

if RUNTIME_SIGNATURE != EXPECTED_RUNTIME_SIGNATURE:
    raise RuntimeError(
        "Firma del runtime distinta de la cerrada en v3.0.3. "
        f"Esperada={EXPECTED_RUNTIME_SIGNATURE}; observada={RUNTIME_SIGNATURE}"
    )
print("Firma Retrieval v3.0.3 confirmada.")


Runtime signature: c34acdb5692df44c35b7e12a9d2c3cbb6061aa37eff24d5157b93c4c44adf86f


,filename,sha256,size_bytes
0,rag_document_index.parquet,e07689220a69ffc1a1182167b1f8f12d0f7dbc6255d1f6...,1360072
1,rag_retrieval_section_index.parquet,b5a0e9ed75dd4387b794a97cc13bb7780587a8dea18a6e...,694827
2,rag_chunk_index.parquet,64197ba81ebd113d38e2bc2906861cc3b6c950b67252a0...,2363344
3,rag_document_embeddings.npy,d27d7c3839507b46ed81658cd258974ad79e1e10098c60...,29312
4,rag_chunk_embeddings.npy,efc17a639cf765d4f1f3a2894f6a2077757115aaaf5c49...,2705024


Firma Retrieval v3.0.3 confirmada.


In [5]:
document_index_df = pd.read_parquet(RUNTIME_DIR / "rag_document_index.parquet")
section_index_df = pd.read_parquet(RUNTIME_DIR / "rag_retrieval_section_index.parquet")
chunk_index_df = pd.read_parquet(RUNTIME_DIR / "rag_chunk_index.parquet")

document_embeddings = np.load(RUNTIME_DIR / "rag_document_embeddings.npy")
section_embeddings = np.load(RUNTIME_DIR / "rag_retrieval_section_embeddings.npy")
chunk_embeddings = np.load(RUNTIME_DIR / "rag_chunk_embeddings.npy")

document_embedding_map_df = pd.read_csv(RUNTIME_DIR / "rag_document_embedding_map.csv")
chunk_embedding_map_df = pd.read_csv(RUNTIME_DIR / "rag_chunk_embedding_map.csv")
lda_k2_descriptors_df = pd.read_csv(RUNTIME_DIR / "rag_lda_k2_descriptors.csv")
lda_k3_descriptors_df = pd.read_csv(RUNTIME_DIR / "rag_lda_k3_descriptors.csv")
micro_descriptors_df = pd.read_csv(RUNTIME_DIR / "rag_microtopic_descriptors.csv")
lda_k2_descriptor_embeddings = np.load(RUNTIME_DIR / "rag_lda_k2_descriptor_embeddings.npy")
lda_k3_descriptor_embeddings = np.load(RUNTIME_DIR / "rag_lda_k3_descriptor_embeddings.npy")
micro_descriptor_embeddings = np.load(RUNTIME_DIR / "rag_microtopic_descriptor_embeddings.npy")
gold_candidates_df = pd.read_csv(RUNTIME_DIR / "rag_goldset_review_candidates.csv")

checks = pd.DataFrame([
    {"criterion":"documentos", "expected":19, "observed":len(document_index_df)},
    {"criterion":"chunks", "expected":1761, "observed":len(chunk_index_df)},
    {"criterion":"retrieval_sections", "expected":877, "observed":len(section_index_df)},
    {"criterion":"document_embeddings", "expected":len(document_embedding_map_df), "observed":len(document_embeddings)},
    {"criterion":"chunk_embeddings", "expected":len(chunk_embedding_map_df), "observed":len(chunk_embeddings)},
])
checks["passed"] = checks["expected"] == checks["observed"]
display(checks)
if not checks["passed"].all():
    raise RuntimeError("El runtime no coincide con el Retrieval cerrado")

,criterion,expected,observed,passed
0,documentos,19,19,True
1,chunks,1761,1761,True
2,retrieval_sections,877,877,True
3,document_embeddings,19,19,True
4,chunk_embeddings,1761,1761,True


## 3. Semántica de alcance y estructura de snapshots (sin planner)

In [6]:
HYPOTHESIS_PROTOTYPES = {'modelo_comunicacional': 'identidad comunicación difusión posicionamiento campaña imagen gubernamental territorio participación', 'modelo_modernizador': 'ciencia tecnología innovación universidad formación capacidades articulación institucional modernización emprendimiento', 'modelo_productivista': 'producción promoción industrial protección sector diversificación empleo cadena de valor mercado infraestructura'}
CONTRASTIVE_PROTOTYPES = {'modelo_comunicacional': 'producción tecnología innovación financiamiento regulación instrumentos productivos sectoriales empleo cadena de valor', 'modelo_modernizador': 'regulación sectorial incentivos beneficios producción tradicional permisos inversiones sectoriales', 'modelo_productivista': 'comunicación institucional identidad difusión ciencia tecnología innovación capacidades'}
PROVINCE_VARIANTS = {
    "Catamarca": ["catamarca", "catamarqueña", "catamarqueño", "catamarqueños"],
    "La Rioja": ["la rioja", "riojana", "riojano", "riojanos"],
    "Santiago del Estero": [
        "santiago del estero", "santiagueña", "santiagueño", "santiagueños"
    ],
}

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9ñ\s\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def parse_pipe(value) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split("|") if x.strip()]

def detect_provinces(query: str) -> list[str]:
    key = normalize_text(query)
    return [
        p for p, variants in PROVINCE_VARIANTS.items()
        if any(normalize_text(v) in key for v in variants)
    ]

def detect_sources(query: str) -> list[str]:
    key = normalize_text(query)
    out = []
    if re.search(r"\bcfi\b", key):
        out.append("CFI")
    if re.search(r"\bsaij\b", key):
        out.append("SAIJ")
    return out

def detect_periods(query: str) -> list[str]:
    years = [int(x) for x in re.findall(r"\b20(?:09|1[0-9])\b", normalize_text(query))]
    periods = []
    for year in years:
        if 2009 <= year <= 2011:
            periods.append("2009–2011")
        elif 2012 <= year <= 2015:
            periods.append("2012–2015")
        elif 2016 <= year <= 2019:
            periods.append("2016–2019")
    return sorted(set(periods))

def semantic_core(query: str, provinces: list[str], sources: list[str]) -> str:
    core = str(query)
    variants = []
    for province in provinces:
        variants.extend(PROVINCE_VARIANTS[province])
    for variant in sorted(set(variants), key=len, reverse=True):
        core = re.sub(
            rf"(?<!\w){re.escape(variant)}(?:s)?(?!\w)",
            " ", core, flags=re.IGNORECASE
        )
    for source in sources:
        core = re.sub(
            rf"(?<!\w){re.escape(source)}(?!\w)",
            " ", core, flags=re.IGNORECASE
        )
    core = re.sub(r"\b20(?:09|1[0-9])\s*[–—-]\s*20(?:09|1[0-9])\b", " ", core)
    core = re.sub(r"\b20(?:09|1[0-9])\b", " ", core)
    core = re.sub(r"\b(?:de|del|en)\s+(?:y|o)\b", " ", core, flags=re.I)
    core = re.sub(r"\b(?:y|o)\s+(?:de|del|en)\b", " ", core, flags=re.I)
    core = re.sub(r"\b(?:de|del|en)\s+(?=[?.,;:]|$)", " ", core, flags=re.I)
    core = re.sub(r"\s+([?.,;:])", r"\1", core)
    core = re.sub(r"\s+", " ", core).strip(" ,.;:-")
    return core if len(normalize_text(core).split()) >= 2 else query

@dataclass(frozen=True)
class QueryIntentSnapshot:
    query: str
    query_type: str
    semantic_core: str
    provinces: tuple[str,...]
    sources: tuple[str,...]
    periods: tuple[str,...]
    hypothesis: str|None
    expected_answer_mode: str
    required_controls: tuple[str,...]
    runtime_signature: str
    signature: str

intent_snapshots={}  # Sólo decisiones registradas por el planificador SS3.


## 4. Recuperación semántico-lexical congelada y composición de evidencias

In [7]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_embedding_lookup = {
    str(chunk_id): index
    for index, chunk_id in enumerate(chunk_embedding_map_df["chunk_id"])
}

chunk_text_column = (
    "policy_view"
    if "policy_view" in chunk_index_df.columns
    else ("text_raw" if "text_raw" in chunk_index_df.columns else "text")
)

def encode_text(text: str) -> np.ndarray:
    return embedding_model.encode(
        [str(text)], normalize_embeddings=True, show_progress_bar=False
    )[0]

def tokenize_bm25(text: str) -> list[str]:
    return [t for t in normalize_text(text).split() if len(t) > 2]

def minmax(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    lo, hi = np.nanmin(values), np.nanmax(values)
    return np.ones_like(values) if hi - lo < 1e-12 else (values - lo) / (hi - lo)

def effective_province_column(frame: pd.DataFrame) -> str:
    if "effective_province" in frame.columns:
        return "effective_province"
    if "display_province" in frame.columns:
        return "display_province"
    return "province"

def allowed_chunk_scope(frame: pd.DataFrame, snapshot: QueryIntentSnapshot) -> pd.DataFrame:
    result = frame.copy()
    province_col = effective_province_column(result)
    if snapshot.sources:
        result = result[result["source"].isin(snapshot.sources)].copy()
    if snapshot.periods and "period" in result.columns:
        result = result[result["period"].astype(str).isin(snapshot.periods)].copy()
    if snapshot.provinces:
        # Un documento regional sólo entra si su chunk fue territorializado
        # a una provincia solicitada. "Regional" no sobrevive como scope efectivo.
        result = result[result[province_col].isin(snapshot.provinces)].copy()
    if "citation_valid" in result.columns:
        result = result[result["citation_valid"].fillna(False)].copy()
    return result

def rank_chunk_pool(
    snapshot: QueryIntentSnapshot,
    doc_ids: list[str],
    semantic_query: str,
) -> pd.DataFrame:
    candidates = chunk_index_df[
        chunk_index_df["doc_id"].isin(doc_ids)
    ].copy()
    candidates = allowed_chunk_scope(candidates, snapshot)
    if candidates.empty:
        return candidates
    candidates = candidates[
        candidates["chunk_id"].astype(str).isin(chunk_embedding_lookup)
    ].copy()

    q = encode_text(semantic_query)
    dense = np.array([
        float(chunk_embeddings[chunk_embedding_lookup[str(r.chunk_id)]] @ q)
        for r in candidates.itertuples(index=False)
    ])

    tokenized = [
        tokenize_bm25(v)
        for v in candidates[chunk_text_column].fillna("")
    ]
    lexical = np.asarray(
        BM25Okapi(tokenized).get_scores(tokenize_bm25(semantic_query)),
        dtype=float,
    )
    quality = (
        candidates["score_quality"].fillna(0.8).to_numpy(dtype=float)
        if "score_quality" in candidates.columns
        else np.full(len(candidates), 0.8)
    )

    candidates["score_dense_v311"] = minmax(dense)
    candidates["score_lexical_v311"] = minmax(lexical)
    candidates["score_quality_v311"] = minmax(quality)
    candidates["score_v311"] = (
        0.68 * candidates["score_dense_v311"]
        + 0.22 * candidates["score_lexical_v311"]
        + 0.10 * candidates["score_quality_v311"]
    )

    return (
        candidates.sort_values("score_v311", ascending=False)
        .groupby("doc_id", as_index=False)
        .head(MAX_CHUNKS_PER_DOCUMENT)
        .sort_values("score_v311", ascending=False)
        .reset_index(drop=True)
    )

def take_unique(source: pd.DataFrame, selected_ids: set[str], n: int) -> pd.DataFrame:
    if source.empty or n <= 0:
        return source.head(0).copy()
    part = source[
        ~source["chunk_id"].astype(str).isin(selected_ids)
    ].head(n).copy()
    selected_ids.update(part["chunk_id"].astype(str).tolist())
    return part

def compose_thematic(pool, snapshot, limit=MAX_EVIDENCE_ITEMS):
    province_col = effective_province_column(pool)
    selected_ids, pieces = set(), []
    for province in snapshot.provinces:
        for source in snapshot.sources or SOURCES:
            cell = pool[
                (pool[province_col] == province)
                & (pool["source"] == source)
            ].sort_values("score_v311", ascending=False)
            part = take_unique(cell, selected_ids, 1)
            if not part.empty:
                pieces.append(part)
    current = sum(len(x) for x in pieces)
    pieces.append(
        take_unique(
            pool.sort_values("score_v311", ascending=False),
            selected_ids, limit - current
        )
    )
    out = pd.concat(pieces, ignore_index=True) if pieces else pool.head(0)
    out["role_v311"] = "PRIMARY"
    return out.head(limit)

def compose_comparative(pool, snapshot, limit=MAX_EVIDENCE_ITEMS):
    province_col = effective_province_column(pool)
    provinces = list(snapshot.provinces)
    if len(provinces) < 2:
        return compose_thematic(pool, snapshot, limit)
    per_province = limit // len(provinces)
    selected_ids, pieces = set(), []
    for province in provinces:
        ppool = pool[pool[province_col] == province].sort_values(
            "score_v311", ascending=False
        )
        local = []
        for source in snapshot.sources or SOURCES:
            part = take_unique(ppool[ppool["source"] == source], selected_ids, 1)
            if not part.empty:
                local.append(part)
        already = sum(len(x) for x in local)
        fill = take_unique(ppool, selected_ids, max(0, per_province - already))
        if not fill.empty:
            local.append(fill)
        pieces.extend(local)
    current = sum(len(x) for x in pieces)
    if current < limit:
        pieces.append(
            take_unique(
                pool.sort_values("score_v311", ascending=False),
                selected_ids, limit - current
            )
        )
    out = pd.concat(pieces, ignore_index=True) if pieces else pool.head(0)
    out["role_v311"] = "PRIMARY"
    return out.head(limit)

def compose_diachronic(pool, snapshot, limit=MAX_EVIDENCE_ITEMS):
    periods = list(snapshot.periods)
    if len(periods) < 2:
        return compose_thematic(pool, snapshot, limit)
    province_col = effective_province_column(pool)
    per_period = limit // len(periods)
    selected_ids, pieces = set(), []
    for period in periods:
        ppool = pool[
            pool["period"].astype(str) == period
        ].sort_values("score_v311", ascending=False)
        local = []
        for province in snapshot.provinces:
            part = take_unique(
                ppool[ppool[province_col] == province],
                selected_ids, 1
            )
            if not part.empty:
                local.append(part)
        for source in snapshot.sources or SOURCES:
            part = take_unique(ppool[ppool["source"] == source], selected_ids, 1)
            if not part.empty:
                local.append(part)
        already = sum(len(x) for x in local)
        fill = take_unique(ppool, selected_ids, max(0, per_period - already))
        if not fill.empty:
            local.append(fill)
        pieces.extend(local)
    current = sum(len(x) for x in pieces)
    if current < limit:
        pieces.append(
            take_unique(
                pool.sort_values("score_v311", ascending=False),
                selected_ids, limit - current
            )
        )
    out = pd.concat(pieces, ignore_index=True) if pieces else pool.head(0)
    out["role_v311"] = "PRIMARY"
    return out.head(limit)

def prototype_query(snapshot: QueryIntentSnapshot) -> str:
    return HYPOTHESIS_PROTOTYPES.get(
        snapshot.hypothesis, snapshot.semantic_core
    )

def contrastive_query(snapshot: QueryIntentSnapshot) -> str | None:
    if snapshot.query_type != "hypothesis_test" or not snapshot.hypothesis:
        return None
    return CONTRASTIVE_PROTOTYPES.get(snapshot.hypothesis)

def compose_hypothesis(
    support_pool: pd.DataFrame,
    contrast_pool: pd.DataFrame,
    snapshot: QueryIntentSnapshot,
    limit: int = MAX_EVIDENCE_ITEMS,
) -> pd.DataFrame:
    asks_contradiction = "contrad" in normalize_text(snapshot.query)

    if asks_contradiction:
        adversarial_n = 6
        adversarial = contrast_pool.head(adversarial_n).copy()
        adversarial["role_v311"] = "ADVERSARIAL"
        used = set(adversarial["chunk_id"].astype(str))
        support = support_pool[
            ~support_pool["chunk_id"].astype(str).isin(used)
        ].head(limit - adversarial_n).copy()
        support["role_v311"] = "SUPPORT_CONTEXT"
        return pd.concat([adversarial, support], ignore_index=True).head(limit)

    support_n = 6
    support = support_pool.head(support_n).copy()
    support["role_v311"] = "SUPPORT"
    used = set(support["chunk_id"].astype(str))
    qualifying = contrast_pool[
        ~contrast_pool["chunk_id"].astype(str).isin(used)
    ].head(limit - support_n).copy()
    qualifying["role_v311"] = "QUALIFYING"
    return pd.concat([support, qualifying], ignore_index=True).head(limit)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (3 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
FINDING_ROBUSTNESS_REFERENCE={
    ("Catamarca","modelo_comunicacional"):{"rank1_rate":0.889,"label":"consistente_condicionada","interpretation":"orientación comunicacional comparativamente distintiva; no exclusiva"},
    ("Santiago del Estero","modelo_modernizador"):{"rank1_rate":0.759,"label":"consistente_condicionada","interpretation":"modernización productiva apoyada, pero combinada con otras orientaciones"},
    ("La Rioja","modelo_productivista"):{"rank1_rate":0.0,"label":"no_consolidada","interpretation":"la caracterización productivista no se consolida en el corpus final"},
}

def finding_robustness(snapshot:QueryIntentSnapshot)->dict:
    if snapshot.query_type!="hypothesis_test" or len(snapshot.provinces)!=1 or not snapshot.hypothesis:
        return {"applicable":False,"label":"no_aplica"}
    ref=FINDING_ROBUSTNESS_REFERENCE.get((snapshot.provinces[0],snapshot.hypothesis))
    return {"applicable":False,"label":"sin_referencia"} if ref is None else {"applicable":True,"province":snapshot.provinces[0],"hypothesis":snapshot.hypothesis,**ref}

In [9]:
@dataclass
class EvidenceItem:
    evidence_id: str
    role: str
    doc_id: str
    title: str
    province: str
    source: str
    period: str
    citation: str
    chunk_id: str
    retrieval_section_id: str
    score: float
    evidence_text: str
    context_window: str

@dataclass
class EvidenceDossier:
    dossier_id: str
    question_id: str
    intent_signature: str
    runtime_signature: str
    query: str
    query_type: str
    hypothesis: str | None
    composition_mode: str
    retrieval_quality: dict[str, Any]
    finding_robustness: dict[str, Any]
    documentary_coverage: dict[str, Any]
    composition_contract: dict[str, Any]
    evidence: list[EvidenceItem]
    coverage_audit: dict[str, Any] | None = None

def safe_text(row: pd.Series) -> str:
    for column in ["text_raw", "policy_view", "text"]:
        value = row.get(column)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def effective_province(row: pd.Series) -> str:
    for column in ["effective_province", "display_province", "province"]:
        value = row.get(column)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def coverage_dossier(qid: str, snapshot: QueryIntentSnapshot) -> EvidenceDossier:
    status_col = "status" if "status" in document_index_df.columns else None
    if status_col:
        status = document_index_df[status_col].fillna("").astype(str).str.lower()
        complete = status.isin({"full_text", "ocr_text"})
        ocr_docs = document_index_df.loc[
            status == "ocr_text", "doc_id"
        ].astype(str).tolist()
    else:
        complete = pd.Series(True, index=document_index_df.index)
        ocr_docs = []

    audit = {
        "documents": int(len(document_index_df)),
        "complete_documents": int(complete.sum()),
        "incomplete_documents": int((~complete).sum()),
        "ocr_documents": ocr_docs,
        "premise_supported": bool((~complete).sum() > 0),
    }
    did = "DOS-" + hashlib.sha256(
        (qid + snapshot.signature + RUNTIME_SIGNATURE).encode("utf-8")
    ).hexdigest()[:16]
    return EvidenceDossier(
        did, qid, snapshot.signature, RUNTIME_SIGNATURE,
        snapshot.query, snapshot.query_type, snapshot.hypothesis,
        "COVERAGE_AUDIT",
        {"label": "auditoría_metodológica", "score": 1.0},
        {"applicable": False, "label": "no_aplica"},
        {"documents": len(document_index_df),
         "sources": sorted(document_index_df["source"].dropna().unique().tolist())},
        {"passed": True, "mode": "COVERAGE_AUDIT", "violations": []},
        [], audit
    )

def composition_diagnostics(
    selected: pd.DataFrame,
    snapshot: QueryIntentSnapshot,
    mode: str,
    target_doc_id: str | None = None,
) -> dict[str, Any]:
    if selected.empty:
        return {"passed": False, "mode": mode, "violations": ["empty_dossier"]}

    province_col = effective_province_column(selected)
    observed_provinces = sorted(
        set(selected[province_col].dropna().astype(str)) & set(PROVINCES)
    )
    observed_sources = sorted(set(selected["source"].dropna().astype(str)))
    observed_periods = sorted(
        set(selected["period"].dropna().astype(str))
        if "period" in selected.columns else set()
    )
    observed_roles = sorted(set(selected["role_v311"].dropna().astype(str)))
    violations = []

    if mode == "EXACT_DOCUMENT":
        if set(selected["doc_id"].astype(str)) != {target_doc_id}:
            violations.append("document_target_impurity")

    elif mode == "THEMATIC_BALANCED":
        miss_p = set(snapshot.provinces) - set(observed_provinces)
        miss_s = set(snapshot.sources) - set(observed_sources)
        if miss_p:
            violations.append("missing_provinces:" + ",".join(sorted(miss_p)))
        if miss_s:
            violations.append("missing_sources:" + ",".join(sorted(miss_s)))

    elif mode == "COMPARATIVE_BALANCED":
        counts = selected[province_col].value_counts()
        required = list(snapshot.provinces)
        vals = [int(counts.get(p, 0)) for p in required]
        if any(v == 0 for v in vals):
            violations.append("missing_comparison_province")
        if vals and max(vals) - min(vals) > 1:
            violations.append("province_imbalance")

    elif mode == "DIACHRONIC_BALANCED":
        miss_periods = set(snapshot.periods) - set(observed_periods)
        if miss_periods:
            violations.append("missing_periods:" + ",".join(sorted(miss_periods)))
        if "Regional" in set(selected[province_col].dropna().astype(str)):
            violations.append("regional_scope_leakage")

    elif mode == "HYPOTHESIS_SUPPORT_QUALIFY":
        if "SUPPORT" not in observed_roles:
            violations.append("missing_support_slot")
        if "QUALIFYING" not in observed_roles:
            violations.append("missing_qualifying_slot")

    elif mode == "HYPOTHESIS_ADVERSARIAL_FIRST":
        if "ADVERSARIAL" not in observed_roles:
            violations.append("missing_adversarial_slot")
        if "SUPPORT_CONTEXT" not in observed_roles:
            violations.append("missing_support_context_slot")

    return {
        "passed": not violations,
        "mode": mode,
        "violations": violations,
        "observed_provinces": observed_provinces,
        "observed_sources": observed_sources,
        "observed_periods": observed_periods,
        "observed_roles": observed_roles,
        "documents": int(selected["doc_id"].nunique()),
        "evidence_items": int(len(selected)),
    }

## 5. Contratos **facticos** y validador — sin autoridad sobre intención

In [10]:
ANSWER_CONTRACTS = {
    "documental": {
        "required_dimensions": ["direct_answer", "exact_document"],
        "forbidden_inferences": [
            "causality", "generalization_beyond_document", "other_documents"
        ],
        "max_paragraphs": 3,
    },
    "thematic": {
        "required_dimensions": [
            "pattern", "province_coverage", "source_diversity", "document_examples"
        ],
        "forbidden_inferences": [
            "raw_frequency_as_prevalence", "causality", "unrepresented_province"
        ],
        "max_paragraphs": 4,
    },
    "comparative": {
        "required_dimensions": [
            "province_A", "province_B", "common_ground", "difference"
        ],
        "forbidden_inferences": ["unbalanced_comparison", "causality"],
        "max_paragraphs": 4,
    },
    "diachronic": {
        "required_dimensions": [
            "early_period", "late_period", "change_or_continuity", "coverage_caution"
        ],
        "forbidden_inferences": [
            "raw_counts_as_historical_change", "causality",
            "regional_scope_as_provincial_evidence"
        ],
        "max_paragraphs": 4,
    },
    "hypothesis_test": {
        "required_dimensions": [
            "support_or_counterorientation", "qualifying_or_adversarial",
            "robustness", "calibrated_conclusion"
        ],
        "forbidden_inferences": [
            "exclusive_model", "causality", "definitive_confirmation",
            "absence_from_silence"
        ],
        "max_paragraphs": 4,
    },
    "coverage": {
        "required_dimensions": ["corpus_status", "ocr_status", "impact_on_answers"],
        "forbidden_inferences": ["invent_missing_documents"],
        "max_paragraphs": 2,
    },
}

@dataclass(frozen=True)
class AnswerContract:
    contract_id: str
    query_type: str
    required_dimensions: tuple[str, ...]
    forbidden_inferences: tuple[str, ...]
    max_paragraphs: int
    sentence_citations_required: bool = True
    evidence_authority_only: bool = True
    fin_token_required: bool = True

def build_answer_contract(dossier: EvidenceDossier) -> AnswerContract:
    if not dossier.composition_contract.get("passed"):
        raise RuntimeError("No se construye AnswerContract sobre un dossier inválido")
    base = ANSWER_CONTRACTS[dossier.query_type]
    payload = {"dossier_id": dossier.dossier_id, "query_type": dossier.query_type, **base}
    signature = hashlib.sha256(
        json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()[:16]
    return AnswerContract(
        "ACT-" + signature,
        dossier.query_type,
        tuple(base["required_dimensions"]),
        tuple(base["forbidden_inferences"]),
        int(base["max_paragraphs"]),
    )

In [11]:
import torch
try:
    from transformers import FineGrainedFP8Config, Mistral3ForConditionalGeneration, MistralCommonBackend
except Exception:
    FineGrainedFP8Config=None
    from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend

LLM_RUNTIME={"ready":False,"model_id":MINISTRAL_MODEL_ID,"device":"","error":""}
EXPLAINER_MODEL=None;EXPLAINER_TOKENIZER=None

def load_ministral_8b()->dict[str,Any]:
    global EXPLAINER_MODEL,EXPLAINER_TOKENIZER
    if EXPLAINER_MODEL is not None and EXPLAINER_TOKENIZER is not None:
        LLM_RUNTIME["ready"]=True;return LLM_RUNTIME
    if not torch.cuda.is_available():
        LLM_RUNTIME["error"]="No se detectó GPU CUDA. Los controles deterministas siguen disponibles, pero no la evaluación LLM."
        return LLM_RUNTIME
    try:
        major,minor=torch.cuda.get_device_capability();kwargs={"device_map":"auto","low_cpu_mem_usage":True}
        if FineGrainedFP8Config is not None and major<9:kwargs["quantization_config"]=FineGrainedFP8Config(dequantize=True)
        token=os.environ.get("HF_TOKEN","").strip();tkwargs={}
        if token:tkwargs["token"]=token;kwargs["token"]=token
        EXPLAINER_TOKENIZER=MistralCommonBackend.from_pretrained(MINISTRAL_MODEL_ID,**tkwargs)
        EXPLAINER_MODEL=Mistral3ForConditionalGeneration.from_pretrained(MINISTRAL_MODEL_ID,**kwargs);EXPLAINER_MODEL.eval()
        LLM_RUNTIME.update({"ready":True,"device":f"{torch.cuda.get_device_name(0)} · compute {major}.{minor}","error":""})
    except Exception as exc:
        EXPLAINER_MODEL=None;EXPLAINER_TOKENIZER=None;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
        LLM_RUNTIME.update({"ready":False,"error":f"No se pudo cargar Ministral: {type(exc).__name__}: {exc}"})
    return LLM_RUNTIME

display(pd.DataFrame([load_ministral_8b()]))

tekken.json: reconstructing file:   0%|          |  0.00B / 16.8MB            

tekken.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/103k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

,ready,model_id,device,error
0,True,mistralai/Ministral-3-8B-Instruct-2512,NVIDIA L4 · compute 8.9,


In [12]:
NARRATIVE_SYSTEM_PROMPT = """
Sos un analista de políticas públicas que redacta una respuesta breve
a partir de un dossier documental ya validado.

AUTORIDAD:
- El dossier es la única autoridad factual.
- QueryIntentSnapshot define alcance y tipo de pregunta.
- AnswerContract define las dimensiones obligatorias.
- No recuperes documentos nuevos y no cambies el scope.

REGLAS:
1. Usá solamente hechos sostenidos por los evidence_ids recibidos.
2. Cada oración sustantiva debe terminar con uno o más IDs: [E001] o [E001,E003].
3. Escribí entre 2 y 4 párrafos breves.
4. No uses títulos, subtítulos, numeraciones ni viñetas.
5. No inventes fechas, cifras, provincias, artículos, fuentes, causalidad ni tendencias.
6. No infieras ausencia a partir del silencio documental: no escribas "no hubo",
   "no existe", "no se documenta" o equivalentes salvo evidencia negativa explícita.
7. En hypothesis_test:
   SUPPORT/SUPPORT_CONTEXT = compatible con la caracterización;
   QUALIFYING = obliga a matizar;
   ADVERSARIAL = otra orientación sustantiva que limita una lectura exclusiva,
   pero no equivale a negación explícita.
8. Si la robustez es condicionada, usá lenguaje calibrado:
   "sugiere", "respalda con matices", "aparece como orientación".
9. No menciones embeddings, RAG, BERTopic, LDA, prompts ni contratos.
10. Terminá exactamente con <FIN>.
""".strip()

def dossier_for_prompt(dossier: EvidenceDossier, contract: AnswerContract) -> dict[str, Any]:
    return {
        "question": dossier.query,
        "query_type": dossier.query_type,
        "hypothesis": dossier.hypothesis,
        "composition_mode": dossier.composition_mode,
        "finding_robustness": dossier.finding_robustness,
        "documentary_coverage": dossier.documentary_coverage,
        "composition_contract": dossier.composition_contract,
        "answer_contract": asdict(contract),
        "evidence": [
            {
                "evidence_id": x.evidence_id,
                "role": x.role,
                "doc_id": x.doc_id,
                "title": x.title,
                "province": x.province,
                "source": x.source,
                "period": x.period,
                "citation": x.citation,
                "text": x.evidence_text,
                "context": x.context_window,
            }
            for x in dossier.evidence
        ],
    }

In [13]:
EVIDENCE_ID_PATTERN = re.compile(r"\[(E\d{3}(?:\s*,\s*E\d{3})*)\]")

CAUSAL_PATTERNS = [
    r"\bcaus[ao]\b", r"\bprovoc[ao]\b", r"\bdebido a\b",
    r"\bcomo consecuencia de\b", r"\bexplica(?:n)?\b",
]
OVERSTATEMENT_PATTERNS = [
    r"\bdemuestra definitivamente\b", r"\bconfirma de manera concluyente\b",
    r"\bprueba que\b", r"\bes exclusivamente\b", r"\bsin lugar a dudas\b",
]
ABSENCE_PATTERNS = [
    r"\bno se (?:documenta|observa|registra|evidencia|encuentra)\b",
    r"\bno (?:hubo|hay|existe|existen)\b", r"\bausencia de\b", r"\bcarece de\b",
]

def sentence_units(text: str) -> list[str]:
    text = str(text).replace("<FIN>", "").strip()
    out = []
    for block in re.split(r"\n+", text):
        block = block.strip()
        if not block:
            continue
        out.extend(
            x.strip()
            for x in re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ])", block)
            if x.strip()
        )
    return out

def unit_type(sentence: str) -> str:
    stripped = re.sub(r"[*_#]", "", str(sentence)).strip()
    if re.fullmatch(r"\d+[.)]?\s*", stripped):
        return "STRUCTURAL"
    if len(stripped.split()) <= 5 and stripped.endswith(":"):
        return "STRUCTURAL"
    if re.match(r"^(?:[-•]|\d+[.)])\s+", stripped) and len(stripped.split()) <= 8:
        return "STRUCTURAL"
    return "CLAIM"

def extract_evidence_ids(sentence: str) -> list[str]:
    ids = []
    for m in EVIDENCE_ID_PATTERN.finditer(sentence):
        ids.extend(x.strip() for x in m.group(1).split(","))
    return ids

def strip_evidence_ids(sentence: str) -> str:
    return EVIDENCE_ID_PATTERN.sub("", sentence).strip()

def numeric_tokens(text: str) -> set[str]:
    text = re.sub(r"^\s*\d+[.)]\s+", " ", str(text))
    return set(re.findall(r"(?<![A-Za-zE_])\b\d+(?:[.,]\d+)?%?\b", text))

def mentioned_sources(text: str) -> set[str]:
    key = normalize_text(text)
    out = set()
    if re.search(r"\bcfi\b", key):
        out.add("CFI")
    if re.search(r"\bsaij\b", key):
        out.add("SAIJ")
    return out

def mentioned_provinces(text: str) -> set[str]:
    key = normalize_text(text)
    return {
        p for p, variants in PROVINCE_VARIANTS.items()
        if any(normalize_text(v) in key for v in variants)
    }

def mentioned_periods(text: str) -> set[str]:
    return {p for p in PERIODS if p in str(text)}

def semantic_alignment(claim_text: str, items: list[EvidenceItem]) -> float:
    if not claim_text.strip() or not items:
        return 0.0
    c = encode_text(claim_text)
    e = embedding_model.encode(
        [x.evidence_text for x in items],
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return float(np.max(e @ c))

def explicit_negative_evidence(items: list[EvidenceItem]) -> bool:
    text = normalize_text(
        " ".join(x.evidence_text + " " + x.context_window for x in items)
    )
    return any(
        re.search(p, text)
        for p in [r"\bno\b", r"\bsin\b", r"\bausencia\b", r"\bcarece\b"]
    )

def validate_sentence(
    sentence: str, dossier: EvidenceDossier, contract: AnswerContract
) -> dict[str, Any]:
    kind = unit_type(sentence)
    if kind == "STRUCTURAL":
        return {
            "sentence": sentence, "unit_type": kind, "claim_text": sentence,
            "evidence_ids": [], "roles": [], "semantic_alignment": np.nan,
            "status": "STRUCTURAL", "reasons": [],
        }

    emap = {x.evidence_id: x for x in dossier.evidence}
    ids = extract_evidence_ids(sentence)
    reasons = []
    if not ids:
        reasons.append("missing_evidence_id")
    unknown = [x for x in ids if x not in emap]
    if unknown:
        reasons.append("unknown_evidence_id:" + ",".join(unknown))
    refs = [emap[x] for x in ids if x in emap]
    claim = strip_evidence_ids(sentence)
    alignment = 0.0

    if refs:
        etext = " ".join(
            x.evidence_text + " " + x.context_window + " " + x.title + " " + x.period
            for x in refs
        )
        unsupported = numeric_tokens(claim) - (
            numeric_tokens(etext) | numeric_tokens(dossier.query)
        )
        if unsupported:
            reasons.append("unsupported_number:" + ",".join(sorted(unsupported)))

        cp = mentioned_provinces(claim)
        ep = {x.province for x in refs if x.province in PROVINCES}
        if cp and not cp.issubset(ep):
            reasons.append("unsupported_province")

        cs = mentioned_sources(claim)
        es = {x.source for x in refs if x.source}
        if cs and not cs.issubset(es):
            reasons.append("unsupported_source")

        ct = mentioned_periods(claim)
        et = {x.period for x in refs if x.period}
        if ct and not ct.issubset(et):
            reasons.append("unsupported_period")

        alignment = semantic_alignment(claim, refs)
        if alignment < MIN_SENTENCE_EVIDENCE_COSINE:
            reasons.append("low_semantic_alignment")

    norm = normalize_text(claim)

    if any(re.search(p, norm) for p in CAUSAL_PATTERNS):
        reasons.append("unsupported_causality")
    if any(re.search(p, norm) for p in OVERSTATEMENT_PATTERNS):
        reasons.append("hypothesis_overstatement")
    if any(re.search(p, norm) for p in ABSENCE_PATTERNS):
        if not refs or not explicit_negative_evidence(refs):
            reasons.append("unsupported_absence_inference")

    if (
        dossier.query_type == "hypothesis_test"
        and dossier.finding_robustness.get("label")
        in {"consistente_condicionada", "no_consolidada"}
        and re.search(r"\b(?:confirma|demuestra|prueba)\b", norm)
    ):
        reasons.append("calibration_overstatement")

    roles = sorted({x.role for x in refs})
    adversarial_language = any(
        token in norm
        for token in ["contrad", "matiz", "limita", "otra orientacion",
                      "no exclusiva", "heterogene"]
    )
    if dossier.query_type == "hypothesis_test" and adversarial_language:
        if not (set(roles) & {"QUALIFYING", "ADVERSARIAL"}):
            reasons.append("role_mismatch_adversarial_claim")

    return {
        "sentence": sentence,
        "unit_type": kind,
        "claim_text": claim,
        "evidence_ids": ids,
        "roles": roles,
        "semantic_alignment": alignment,
        "status": "ACCEPT" if not reasons else "REJECT",
        "reasons": reasons,
    }

def validate_narrative(
    text: str, dossier: EvidenceDossier, contract: AnswerContract
) -> pd.DataFrame:
    return pd.DataFrame([
        validate_sentence(s, dossier, contract)
        for s in sentence_units(text)
    ])

In [14]:
def accepted_text(vdf: pd.DataFrame) -> str:
    if vdf.empty:
        return ""
    return "\n\n".join(
        vdf.loc[vdf["status"] == "ACCEPT", "sentence"].tolist()
    )

In [15]:
def render_citations(text:str,dossier:EvidenceDossier)->str:
    mapping={x.evidence_id:x.citation for x in dossier.evidence}
    def repl(m):
        ids=[x.strip() for x in m.group(1).split(",")];c=[mapping[x] for x in ids if x in mapping]
        return " "+" ".join(c) if c else ""
    return EVIDENCE_ID_PATTERN.sub(repl,str(text))

def render_calibration(dossier:EvidenceDossier)->str:
    parts=["Evidencia recuperada: "+dossier.retrieval_quality.get("label","sin etiqueta").replace("_"," ")]
    if dossier.finding_robustness.get("applicable"):parts.append("robustez del patrón: "+dossier.finding_robustness.get("label","sin etiqueta").replace("_"," "))
    sources=dossier.documentary_coverage.get("sources",[])
    if sources:parts.append("fuentes: "+" + ".join(sources))
    return "**Calibración.** "+"; ".join(parts)+"."

## 6. Cache de retrieval: sólo optimización computacional

In [16]:
import threading
from functools import lru_cache
import gradio as gr
from collections import Counter
from time import perf_counter

LIVE_AUDIT_LOG = AUDIT_DIR / 'gradio_live_queries_v313.jsonl'
LIVE_AUDIT_LOCK = threading.Lock()
LIVE_MAX_DOCUMENTS = 8
LIVE_MAX_EVIDENCES = 10
DOC_TEXT_COLUMN = next((c for c in ['policy_view','text_raw','text_excerpt','text','document_text'] if c in document_index_df.columns), None)
DOC_PROVINCE_COLUMN = next((c for c in ['base_province','province','display_province'] if c in document_index_df.columns), None)
assert DOC_PROVINCE_COLUMN and 'doc_id' in document_index_df.columns
DOC_EMB_LOOKUP = {str(r.doc_id):int(r.embedding_row) for r in document_embedding_map_df.itertuples(index=False)}


# No usamos el índice Gold como fuente de evaluación de consultas nuevas.
DOC_ALL = document_index_df.copy().reset_index(drop=True)
DOC_ALL['_title'] = DOC_ALL.get('title', pd.Series([''] * len(DOC_ALL))).fillna('').astype(str)
def _doc_retrieval_text(row):
    chunks = [str(row.get('title',''))] * 3
    if DOC_TEXT_COLUMN:
        value=row.get(DOC_TEXT_COLUMN,'')
        if isinstance(value,str): chunks.append(value[:3500])
    return ' '.join(chunks)

DOC_ALL['_retrieval_text'] = DOC_ALL.apply(_doc_retrieval_text, axis=1)
DOC_BM25 = BM25Okapi([tokenize_bm25(x) or ['vacío'] for x in DOC_ALL['_retrieval_text']])
DOC_TITLE_BM25 = BM25Okapi([tokenize_bm25(x) or ['vacío'] for x in DOC_ALL['_title']])

# Cache de embeddings de consulta; se usa en retrieval y validación.
@lru_cache(maxsize=384)
def _cached_embed(text: str):
    return embedding_model.encode([str(text)], normalize_embeddings=True, show_progress_bar=False)[0]

def encode_text(text: str) -> np.ndarray:
    return _cached_embed(str(text))

@lru_cache(maxsize=100)
def _chunk_bm25(ids: tuple, texts: tuple):
    return BM25Okapi([tokenize_bm25(t) or ['vacío'] for t in texts])

# Redefinición de rank_chunk_pool: misma fórmula de puntuación y cuotas,
# con reutilización segura del objeto BM25 para un mismo corpus candidato.
def rank_chunk_pool(snapshot, doc_ids, semantic_query):
    candidates = chunk_index_df[chunk_index_df['doc_id'].isin(doc_ids)].copy()
    candidates = allowed_chunk_scope(candidates, snapshot)
    if candidates.empty: return candidates
    candidates = candidates[candidates['chunk_id'].astype(str).isin(chunk_embedding_lookup)].copy()
    if candidates.empty: return candidates
    qvec = encode_text(semantic_query)
    ids = tuple(candidates['chunk_id'].astype(str).tolist())
    texts = tuple(candidates[chunk_text_column].fillna('').astype(str).tolist())
    bm = _chunk_bm25(ids,texts)
    dense = np.asarray([
        float(chunk_embeddings[chunk_embedding_lookup[str(r.chunk_id)]] @ qvec)
        for r in candidates.itertuples(index=False)
    ])
    lexical = np.asarray(bm.get_scores(tokenize_bm25(semantic_query)), dtype=float)
    quality = (candidates['score_quality'].fillna(.8).to_numpy(dtype=float)
               if 'score_quality' in candidates.columns else np.full(len(candidates),.8))
    candidates['score_dense_v311']=minmax(dense)
    candidates['score_lexical_v311']=minmax(lexical)
    candidates['score_quality_v311']=minmax(quality)
    candidates['score_v311']=(.68*candidates['score_dense_v311']+.22*candidates['score_lexical_v311']+.10*candidates['score_quality_v311'])
    return (candidates.sort_values('score_v311',ascending=False)
        .groupby('doc_id',as_index=False).head(MAX_CHUNKS_PER_DOCUMENT)
        .sort_values('score_v311',ascending=False).reset_index(drop=True))

# Batch: evita volver a computar el embedding de los mismos textos referenciados
# en cada oración. Sólo modifica el costo de la verificación, NO sus umbrales.
SEMANTIC_CLAIM_CACHE = {}
SEMANTIC_EVIDENCE_CACHE = {}

def _prewarm_validation_embeddings(text, dossier):
    claim_texts = [strip_evidence_ids(s) for s in sentence_units(text) if unit_type(s)=='CLAIM']
    new_claims = list(dict.fromkeys(t for t in claim_texts if t and t not in SEMANTIC_CLAIM_CACHE))
    new_evidence = list(dict.fromkeys(x.evidence_text for x in dossier.evidence if x.evidence_text and x.evidence_text not in SEMANTIC_EVIDENCE_CACHE))
    if new_claims:
        vectors=embedding_model.encode(new_claims,normalize_embeddings=True,show_progress_bar=False,batch_size=32)
        SEMANTIC_CLAIM_CACHE.update(zip(new_claims,vectors))
    if new_evidence:
        vectors=embedding_model.encode(new_evidence,normalize_embeddings=True,show_progress_bar=False,batch_size=32)
        SEMANTIC_EVIDENCE_CACHE.update(zip(new_evidence,vectors))
    # Acotamos RAM: la persistencia histórica queda en JSONL, no en la cache.
    if len(SEMANTIC_CLAIM_CACHE)>500: SEMANTIC_CLAIM_CACHE.clear()
    if len(SEMANTIC_EVIDENCE_CACHE)>1000: SEMANTIC_EVIDENCE_CACHE.clear()

def semantic_alignment(claim_text,items):
    if not claim_text.strip() or not items: return 0.0
    q=SEMANTIC_CLAIM_CACHE.get(claim_text)
    if q is None: q=encode_text(claim_text)
    e=np.stack([
        SEMANTIC_EVIDENCE_CACHE.get(x.evidence_text)
        if x.evidence_text in SEMANTIC_EVIDENCE_CACHE
        else encode_text(x.evidence_text)
        for x in items
    ])
    return float(np.max(e @ q))

_original_validate_narrative = validate_narrative
def validate_narrative(text,dossier,contract):
    _prewarm_validation_embeddings(text,dossier)
    return _original_validate_narrative(text,dossier,contract)

print('Caches BM25 / embeddings y validación por lotes disponibles.')

Caches BM25 / embeddings y validación por lotes disponibles.


In [17]:
def _document_allowed(row,snapshot):
    if snapshot.sources and str(row.get('source','')) not in snapshot.sources: return False
    if snapshot.periods and str(row.get('period','')) not in snapshot.periods: return False
    province=str(row.get(DOC_PROVINCE_COLUMN,''))
    regional=(str(row.get('document_scope',''))=='regional' or province=='Regional' or str(row.get('display_province',''))=='Regional')
    if regional: return False
    if snapshot.provinces and province not in snapshot.provinces: return False
    return True

def _seed_dynamic_docs(df,snapshot,limit):
    df=df.sort_values('live_score',ascending=False)
    used=set(); positions=[]
    def keep(sub,n=1):
        if n<=0 or len(positions)>=limit: return
        for ix in sub.index:
            if ix in used: continue
            used.add(ix);positions.append(ix)
            if len(positions)>=limit: return
            n-=1
            if n<=0: return
    if snapshot.query_type in {'thematic','comparative'}:
        provinces=list(snapshot.provinces) or PROVINCES
        for p in provinces:
            if snapshot.query_type=='thematic':
                for source in snapshot.sources or SOURCES:
                    keep(df[(df[DOC_PROVINCE_COLUMN]==p)&(df['source']==source)],1)
            else:
                keep(df[df[DOC_PROVINCE_COLUMN]==p],2)
    if snapshot.query_type=='diachronic':
        for period in snapshot.periods:
            keep(df[df['period'].astype(str)==period],2)
    keep(df,max(0,limit-len(positions)))
    return df.loc[positions[:limit]].copy().reset_index(drop=True)

def dynamic_document_candidates(snapshot,limit=LIVE_MAX_DOCUMENTS):
    # BM25 documental preconstruido al iniciar la app; sólo cambia el query.
    mask=DOC_ALL.apply(lambda r:_document_allowed(r,snapshot),axis=1)
    df=DOC_ALL.loc[mask].copy()
    df=df[df['doc_id'].astype(str).isin(DOC_EMB_LOOKUP)].copy()
    if df.empty: return df
    q=encode_text(snapshot.semantic_core)
    rows=df.index.to_numpy(dtype=int)
    dense=np.array([
        float(document_embeddings[DOC_EMB_LOOKUP[str(doc_id)]] @ q)
        for doc_id in df['doc_id'].astype(str)
    ])
    tokens=tokenize_bm25(snapshot.semantic_core)
    lexical=np.asarray(DOC_BM25.get_scores(tokens),dtype=float)[rows]
    title=np.asarray(DOC_TITLE_BM25.get_scores(tokens),dtype=float)[rows]
    df['live_score']=.65*minmax(dense)+.30*minmax(lexical)+.05*minmax(title)
    if snapshot.query_type=='documental':
        return df.sort_values('live_score',ascending=False).head(limit).reset_index(drop=True)
    return _seed_dynamic_docs(df,snapshot,limit)

def dynamic_compose(snapshot,doc_ids,limit=LIVE_MAX_EVIDENCES):
    if snapshot.query_type=='documental':
        target=doc_ids[0]
        pool=rank_chunk_pool(snapshot,[target],snapshot.semantic_core)
        out=pool.head(limit).copy();out['role_v311']='PRIMARY'
        return out,'EXACT_DOCUMENT',target
    if snapshot.query_type=='thematic':
        pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
        return compose_thematic(pool,snapshot,limit),'THEMATIC_BALANCED',None
    if snapshot.query_type=='comparative':
        pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
        return compose_comparative(pool,snapshot,limit),'COMPARATIVE_BALANCED',None
    if snapshot.query_type=='diachronic':
        pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
        return compose_diachronic(pool,snapshot,limit),'DIACHRONIC_BALANCED',None
    if snapshot.query_type=='hypothesis_test':
        support=rank_chunk_pool(snapshot,doc_ids,prototype_query(snapshot))
        alt=contrastive_query(snapshot)
        contrast=rank_chunk_pool(snapshot,doc_ids,alt) if alt else support.head(0).copy()
        out=compose_hypothesis(support,contrast,snapshot,limit)
        mode=('HYPOTHESIS_ADVERSARIAL_FIRST' if 'contrad' in normalize_text(snapshot.query)
              else 'HYPOTHESIS_SUPPORT_QUALIFY')
        return out,mode,None
    pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
    out=pool.head(limit).copy();out['role_v311']='PRIMARY'
    return out,'GENERIC',None

def dossier_from_documents(query_id,snapshot,docs,limit=LIVE_MAX_EVIDENCES):
    if snapshot.query_type=='coverage':
        return coverage_dossier(query_id,snapshot)
    if docs.empty: raise ValueError('No hay documentos en el ámbito consultado.')
    doc_ids=docs['doc_id'].astype(str).tolist()
    selected,mode,target=dynamic_compose(snapshot,doc_ids,limit)
    if selected.empty: raise ValueError('No se recuperaron fragmentos con localizador verificable.')
    composition=composition_diagnostics(selected,snapshot,mode,target_doc_id=target)
    items=[]
    for i,(_,r) in enumerate(selected.iterrows(),1):
        items.append(EvidenceItem(
            f'E{i:03d}',str(r.get('role_v311','PRIMARY')),str(r['doc_id']),
            str(r.get('title','')),effective_province(r),str(r.get('source','')),
            str(r.get('period','')),str(r.get('citation','')),str(r.get('chunk_id','')),
            str(r.get('retrieval_section_id','')),float(r.get('score_v311',0)),
            safe_text(r)[:MAX_CONTEXT_CHARS_PER_EVIDENCE],str(r.get('context_window','') or '')[:600]
        ))
    coverage={
        'documents':len({x.doc_id for x in items}),
        'provinces':sorted({x.province for x in items}),
        'sources':sorted({x.source for x in items}),
        'periods':sorted({x.period for x in items}),
        'roles':sorted({x.role for x in items}),
    }
    did='DOS-'+hashlib.sha256((query_id+snapshot.signature+'|'.join(x.chunk_id for x in items)).encode()).hexdigest()[:16]
    return EvidenceDossier(did,query_id,snapshot.signature,RUNTIME_SIGNATURE,
        snapshot.query,snapshot.query_type,snapshot.hypothesis,mode,
        {'label':'experimental_dynamic_not_gold_validated',
         'candidate_documents':len(doc_ids),'frozen_document_universe':True,'live_ranking':True},
        finding_robustness(snapshot),coverage,composition,items,None)


def evidence_rows(dossier):
    return pd.DataFrame([
        {'ID':x.evidence_id,'Rol':x.role,'Documento':x.doc_id,'Provincia':x.province,
         'Fuente':x.source,'Período':x.period,'Localizador':x.citation,
         'Fragmento':x.evidence_text[:620]}
        for x in dossier.evidence
    ])

In [18]:
def _balanced_by_scope(pool, provinces, limit):
    if not provinces or pool.empty:return pool.head(0).copy()
    pcol=effective_province_column(pool)
    groups=[pool[pool[pcol]==p].sort_values('score_v311',ascending=False) for p in provinces]
    if any(g.empty for g in groups):return pool.head(0).copy()
    quota=min([limit//len(groups)]+[len(g) for g in groups])
    used=set();chunks=[]
    for g in groups:
        selected=[]
        for source in SOURCES:
            for idx,r in g[g.source==source].iterrows():
                if len(selected)<quota and r.chunk_id not in used:
                    selected.append(idx);used.add(r.chunk_id);break
        for idx,r in g.iterrows():
            if len(selected)>=quota:break
            if r.chunk_id not in used:selected.append(idx);used.add(r.chunk_id)
        chunks.append(g.loc[selected])
    result=pd.concat(chunks,ignore_index=True)
    result['role_v311']='PRIMARY'
    return result

# Selección documental condicionada exclusivamente por operación calculada por PySS3.
# Las cuotas y localizadores son controles de evidencia, no planificación de intención.
def dynamic_compose(snapshot,doc_ids,limit=LIVE_MAX_EVIDENCES):
    op=snapshot.query_type
    if op=='documental':
        pool=rank_chunk_pool(snapshot,[doc_ids[0]],snapshot.semantic_core)
        output=pool.head(limit).copy();output['role_v311']='PRIMARY'
        return output,'EXACT_DOCUMENT',doc_ids[0]
    if op=='comparative':
        pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
        return _balanced_by_scope(pool,list(snapshot.provinces),limit),'COMPARATIVE_BALANCED',None
    if op=='thematic':
        pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
        return compose_thematic(pool,snapshot,limit),'THEMATIC_BALANCED',None
    if op=='diachronic':
        pool=rank_chunk_pool(snapshot,doc_ids,snapshot.semantic_core)
        return compose_diachronic(pool,snapshot,limit),'DIACHRONIC_BALANCED',None
    if op=='hypothesis_test':
        support=rank_chunk_pool(snapshot,doc_ids,prototype_query(snapshot))
        cq=contrastive_query(snapshot)
        counter=rank_chunk_pool(snapshot,doc_ids,cq) if cq else support.copy()
        target='adversarial' if 'ss3_evidence_focus_adversarial' in snapshot.required_controls else 'qualifying'
        if target=='adversarial':
            a=counter.head(min(6,limit)).copy();a['role_v311']='ADVERSARIAL'
            seen=set(a.chunk_id.astype(str))
            b=support[~support.chunk_id.astype(str).isin(seen)].head(max(0,limit-len(a))).copy();b['role_v311']='SUPPORT_CONTEXT'
            out=pd.concat([a,b],ignore_index=True);mode='HYPOTHESIS_ADVERSARIAL_FIRST'
        else:
            a=support.head(min(6,limit)).copy();a['role_v311']='SUPPORT'
            seen=set(a.chunk_id.astype(str))
            b=counter[~counter.chunk_id.astype(str).isin(seen)].head(max(0,limit-len(a))).copy();b['role_v311']='QUALIFYING'
            out=pd.concat([a,b],ignore_index=True);mode='HYPOTHESIS_SUPPORT_QUALIFY'
        return out,mode,None
    raise ValueError('Operación SS3 no soportada por el renderer documental: '+op)

_CONTRACT_EVIDENCE_ONLY=composition_diagnostics

def composition_diagnostics(selected,snapshot,mode,target_doc_id=None):
    result=_CONTRACT_EVIDENCE_ONLY(selected,snapshot,mode,target_doc_id)
    if snapshot.query_type=='diachronic' and snapshot.periods and not selected.empty:
        observed=set(selected.period.astype(str))
        missing=set(snapshot.periods)-observed
        result['observed_periods']=sorted(observed)
        if missing:result['violations'].append('missing_periods:'+','.join(sorted(missing)))
    result['passed']=not result['violations']
    return result


## 7. Corpus sintético ampliado — datos independientes de las pruebas

In [19]:
import io, base64, zlib, hashlib
SS3_BOOTSTRAP_PAYLOAD = 'eNrVnc9y3DiSh+8bse/A8LnaYQHWv5NCI7ujvdE99tgeR8xJgSJRJXpIohokNZZjD/sO+wQ++uBDh27eQ0d0vcm+wL7CZgJgFUtSsUAwKakPHa2SylX5gQDyh0Qi8X//83sp8kUmz9Nk8mstyypVxUQtpBbmJ52W/ywnSZrLooTX5WSuVb3ANy+ErlLznlLVOpaTTExldq7lZSr/dV5WoqrLf/+3929/ePH67Idnez/sTd7J5W8iW36JMlVG8K+X3+Iaf4SvjbSc15kookyU0UJly29VGuOPWl2mRZyKTJZRIiP4WRZVegn/ShZRXYioUDoX8KdK6jwtRCKeThIV12BvJbJJWgBQVaOZIjsXVaXTqXk1gU9O6tj8+OPb019e/rD+V2DtpNIiLSZ/ef36/bv3b0/fnJ/+/f1Pr9++Ov35/N0//vr+p5fvX51NjHGl/ehCVecXdS4K1wAyabMzT3b3Y5wurwvEhd+kRVKXYDYyZlGpZtW/hJZD6SsZXxQqU/Ore6DnvvTyUmV1G36GePYXYDEYvLyeQ7/4E9Ez6PdvtIzTEuh/rZdfo7L+KCtgj1VRSRh5QJMBQ6zht9AYM6nTREUC8ZN0VpeG3rHE5guxceD751rkovREjlWe1wW03ZYez4ipWTh1Jq+QcKbQNvzj3HyhJ+jGX+4BlAeCwm9lCf3fdOAoTiXOctGV6+dqvvxG0ptpcTn05j9+P6uXX9B6fFiirhQQiQSA8wWMRKA2wzpJy4UqJHKi6bq2T9OMa2gdkchCmCn9UmRKRwL8ymqqg8n+hGAW58TsjIBdleuhC+/T6Ry6Q4n9IV9oWYCjTfG9j5Kf0/BrWdZZJRAbfoNjRKqolDF+mMg8wdN8IeJqdOjn0OFffoJPhJENxqLTMXBlar2SmqUoThzXrzV8Jg5+0EbQPHEsE6nNpFbiK/g6q2JQOcUrP79Yfp9maax+WOj00t+J7ZronhM3BKNoCPjVTBX46K+iPI21ivXya2Lf6royaDvPFpjVMF6K+T2wcyL2sp6WaZKiem2PDGiUtIxx2ntk5PvQ/V+kYEUxr5ffUHbBkF7pN/RxbiibLj4TsXkl8enCV1yk09R1ei0zUB6XOA78JT6Fkt8nbg82Rnt4y34KRUDdInyUFvFdCjy+FjmAMfOzgtGcfgZNCK1RyU+N8HErFvSRSs9FkX6+b51/QMzK+rKOq+6p8XgvvHvS9LSQh0bT/w3XLrKsxDSTMaiyT6CuRG4U3caAhqV2LDXoNTXV9yjlD4mRWTjyfSp4amw+CHtc4U7LeoTCvYAP1xh1EmY9njS+ZoHvhxYwQUQ3Jxm19iBC/YgYnIWA35Mwp2blgazjC3Fa0mPozk/eyfnyuwsTlyt5tJDQSQscv5Poj99NDMo6pDJXkUjyFH6qdA/ZffKEYOY6JsZnw/G9VbZvA3Q6aeoG4AQN4CuqH18D7D1bq5RY5NMUx/NcZNZvwbA38ajkQloZbbQZriZWywr/gPoJidDeezZJ5CUZPCOA36K8T0i8GDEvH8rrIcVPhndyUmrcIv0gdTpLY+POohl8szG/FIUdpODBapQgG5MYPMhMuK20WSpBrAk/YU4QX9nbI20ARtcAvjL98TUCJ22EbtH+dJhoJyVnGwvRdFFnZtqKs+WXusQXdiUylYU0AVedWqZ+op1oumOk4CwM3E+0nwyUssSsPJh1p2h/LKRnr3+xGS8v0hmMRBy2X2AVUgHYmYDFttAx+qJ3AgS3mCsjPl+WFa62kXODTZQKmsFNXT7xJdyLE5i9cyknt51X66+EqQ4OmFEDe7qxDeRbM/WoyJwc2d9xPRg2M7uiC1XAKF7UBiExwuxS6nnTKZN0/Qpn5XzaXnUax7RAN+S3LXyDdsMhbZAyYlJGTNozwrRJfYdTGhWe0z9mryjTBvXmdD0ibzu7xVmLpsKY1Sn2QpvVU6bzGh5WlMuPSsub/dbPT0UvP6UwW2P8AWRcYj4HJgrZNHCc1anebAZVV/BSnsvZDJXfVVezcOJmYcObxTv4RNE03XMhdetwgtbxj01RtE+3KqBtn+drVb9uE6vySnj8Ub78gl1D98r/LG0kLJdJmkB3ulA5iJ+vhcToTZu8rBdSZ/bnbujnxNDMG9onJhUEfFeEalRm7s28JRQVhLnTKdJiYkrMG/BhZs+hTAsQb+m8EBo8JCxoVGmmulLCQBfFZ1FuPlRZugwJUhm/TwzIhgLSy3ZqRD4YcRSZTouJmShnCpYkwmo4nGyaxa6dVM9+fNXEU9dR1Henr/5jvVNGJMsPiMnYQLJRZTg1LB/+GEllNy0fJqA8cZ5jJQyjWe1cBsaBxOcIvIOYKp3AKPXS2JPGmbQ6MEJoNcWfT550SqT/bHuYjpY4JG4J1rMlfGU1YWt0z2DUDcL7Noi3kiZskm73TNskR61Fao7uCgZ4Lq7AwZqVeGny8YE7V4nI8IVslhjeu7+wxFhkZvshXl7nysgxQH0aKK2PiPlZIL/X5m9v9p0qmxqfB+Jv2/7tTbzT/dESY+rLS5CQNVq4+hqXsYYbwSBJwJzETngtT2B3EqjF9jExHBsCRy+0qfH4ILxRRDYpoktMsQPSeRLrnFJgBMuzZrXQLaShR07FR2XPEMawepa4ejbnsGOXprF1PG4T3VTZCY6TeXD2ktVDmHfMQcTo3O8Re4jsAczbZSgp7Z7p0G9qmWBmO84tupT2ECssiRdy+Zty+1zZZqTfxHZ8pLlZcrfilycUUpwqFcG1ARvQBt4lDMLboXOGI24KPqQp/CsahDdGp8MmbQzc+HxyZj7dFDMw+TYxTmXr1clqbQ4zocRN4H7qe2IDUiZbIdWwxEEPUT590kOjUKYxmOSUQGQfwe2Hu0tkExPzUOItGtsPcqdPo8rDeXVqs1PeStyZgvl5+c0MXlmJBQY+61iW5iQegoPV4lKWy2tvDwcTg9mKQm32GV5fiBLaaZZaTZBhwvLym0rU00mSivhCK3iwN8NN679QFuSw4CwEfLdTe+zYvDe2tx8LQ7+dUjkaPc7bL2QJemtq2NGiMi2rVdpCH0eFAVX4s543JwGT5VczPcBqvAYGczCoxXk7gbjFyYg52U5O703WR83Jd3J67av2Zbwz6380zHVCCsy6ME71PJXwRVFRy8ubOWRuW2Pr1upVFOMCRtrKNGlRC/ikkw24O3NeW3CcGI71g/OI5PhQdnZTakTe9/l5xHN8IDvnVlpITPB4laArsCnrpYxraZcA+Jjw/IZCk62pJj7Zb//Um+s5MRcL5NqZct8ecxuZ9SPC8BCYvsdhe0wmtHz79nxkI9nLKFduXLFnz47/97/+mz2DhSC82GP2xT7Qwv8O7KtjV6ylwKP+bofRZ8f0iaeW2yeGZdSwu8/DPhQqJ0X1Lj3zxH/aoSU+aOXLaVGUTp2ZIQhyXcsYM4hcwMVKeLDbFiC6MnIuRXe/mXPsn17o7yoPiLHZCNjeW54Ph81HwPY85tpD19JSH64Sk8oKd4bgS5on0WwNlaskwBuFgbamDqYFzGJYdk5H8iO4N/OU+/mkQ2JKFkbpI3R9cDv7MTUrD32iPorXh7ZzjqalPVrP0ZcCTFyVsJrbo/bwAGOlF7XJFMwF7vGIXtUfTfAcfHITOCl7qPsjYlQWjLpjx7ObcaswpubjYXz9djp3sO6Yl2iRTS2RFyZkh5ETneZSN77FRu7g18vvWZXmYlVSopRAX1T1peqx2XniKxqPiQEZBeDuYzwPhccH4/WpBeM78ZBSmryKJ69xwxH8CVZSVrqpAGHP2MTN42ideS/6lVOcoFKyhzcFRmjcUfvVgWDo3JepPdt2Y0HQ5VrJikLYZmDBzeC/5fgnaAYe3Ax+25ChTbBDOZO2Qjs140JMM7MV7yrS6npR1TAPXIDfWl7rdNeeJOb9iTwtXOWsWS2bQ2Qin+Kc9wkWHfnN+NsOT0VWD8LSsp60PkVfgrA7+zktM+/9hH1KMAZRd832pNS4PfeLAmPBPlRKZvSVdt6ujBGrwic39nnsWtguJhI5RlSZctfdJFOQgVKEmYnpOBHdeHFnKuCf/vHGXWoD82oFAkuY4Dmuf50sU+sC4Yktiw89MFF4UYn39qS752Slz55OLq4WqsLKV+U5jODqbv9z402Eu+kOmwVjbwvfeJDe9WRHJ+VhpD4hHA/kW35nZF6ciP9Sl7hD5Na6mc3brtJpJtF+3Clafi2wAD58WqseWyw0zDDwTz63hbdXevot7Fvz8S1sRozNSLH9vdAjQOfU6Lv80u2B3XZOI/PalIqfzDLZLIxNYRNcHpfgueCxoNZC/zRLs4A4z0ybNYmIFDi8zR3+k8AZjRPzs378nkl//cA3QyWjI/PeyLvT/R4xMGZk/AhL/RpvXIO/JuudfrFQV7jRjyfEYIh/tvuj6IYyGKC4usDIjzYZUEWfmzoCJrLnxNCMCrrHdRwBHpsam5Nh+9+58eDY+61akZ9iuXATdAaetRJWV12ki+V1hRbgW7IaU1xXx+Y8t0jD1PY+MSobhrq17m+YP6Km4wPovKr8BnRWWsYDUy7PxV9aNCID9VS4Za9ZP6hEg8kFcOKRXJewMo6CPiBGZMGIo6plakw+BJNYGdOyHd6oV9ouMDYTl0rjhSGoAuEXUtvdexyeeAhSw4NqipG0D8T0C+OEzUeHxK3AqFvBt/pKX9FIDc5HAPe4Z+NhsVupHCv1hNdblstrvL5MZJgwi82wyi3HhYMy11vCXCVXd+eAdsgViEPVr9xhyJx2RNwCbJQW6JFzGOChqduAj9MG/gmID94GmBZyWqzvOosx5xbjO+tbdYQLiUTgr9bVpxIYyrU2qsw//TA0kn1MzMyImLdeaxfm1KgxOQWmVymVgH5MCmsTR/74/a0r1GCKr5b2buLEHhlLtQvgu8O8zaH9nUJ7EpmwkZoaD1is28rm6zYtaTKD+s/rVKkCrhVYSCv4avE/VUvwwJboluuDmqBT2JPi22sIYBhXy28R7lhNm4NjWsiPCKXcIV1R6Y0xr+5Nw1MlEzhgNgh4JLlOzMiHMtIrc1JCZsvsmn3Xha7lVNy95WS2W+1Y7nFg/QrwywjemadVWPyEMo/ApIX0h/U/sb4Td6ePJsblIbjeB9cfD+7Z6w82S6TJ6RIlPDNtsrowTCTx6A6OX1ssW8vld1hRVCoRtgbx6pp5M/3638Qaq0upxVxOCjk3xUPO3bLmVrUw+z7SC0I+2AwRSuSeNzx14d9ZZGWsVuDUD97ruqcO/Jv1s8YBtzd5/QwjVKDkMvmLGAmVSWrvGbb5AyqTLiPMzQLYMkVdwBCWn/BO+eW1CzE12WE9boIKaARG3AhsxEbwDzB1NMQdVcTGags+Zlv0CDV1tMYdNbXGaQ3MQTmtYeDa1XgBVmT29g6cyKumIOaNxHZ8wABpyxT33K9dUXdBcmJIRgLpF0xaA95dKmwsRk7CuDV4tMbq9lu0WK0rW5qcMHMk7tc61baAicZKUip20YBoeb2AeUjZUhJWxUn/oi0nft3zOTEjI2P0S8FvUXbMu9SUnI7SNzb0IKT75ki5qd5uByPIeBhiRh5/gi+Km3lydZ6z//1zq7F4t4TeJwZiQ4B632/hN9FQM/KBD83zWosGbpvyo8VqlfBoFqHWK0ylNnlXrXNl6wB80ZSaFLMU3Hi+UQV7HMF7QIzNSLHHlrjU9JyWfnRRS8t/aDr9O4l3KCUYssnxJJDQYLTR7nIVWi9UdCGyDP5UF81B+/77pWHMh8TMjIDZLxWxg7db8lIjcwLkrSmJwZEbWsqjzXqhWNK0BqFkBDuIpfW1ja0l6zrxxfPKCQ/Ne0RMxQZQed814aH9qLn4EK4+l0zcO9uxvWrZXtBiTi9oucAUrKISxc2VZTOOTJ6WeTP0wsRcmAQOqMeVbjuU7jExIqNFHEn7UlNz8gdLpIZJQd0FKS+ktRveI22hWlGXq/O/zYF0m/YNvqSpxTC6Bqa7OuNDc0fKUNSRdS8xMacgHlvrkjLb/bwzc1mWwFL+Jsr1+uwtmg0j57OYppm7hytbn6rEqmdG2cECt2cA1y9CRnfTxYem+AYFZE95261miRk5EeNuPdvtW0ixcPftA5jqKv2mIMOvwO19maco3bCUXiU/GStjvEQF87pbxc3cNEOmYSm30k2ixGA0QiFLDMcJ4GjVLBUgvNMmQpytsjxcQV+QYdC1Fsb+zYLoTqiB7SWoeGEpep3OrC4knny8q4BP8yfCrW/HyCgY/bf2HoSRUzD67eyt+e5yDGMhbl4YYi/GLKW7ZnZHHZPb9zNv8LWR7nIKayRGjMQ6kDwLlvRg6+iO1GC881n5FJjth3ZrvhwLDTMF3uEJq8qOMywJhOG2CrqbuaLerdmNup6LxN1o4u0CvJA4MRILQ9oVtWiNqnbUYjwOHsDRMzThO1PQorX2/ytpbslpKpGj9NCm8EW+ytuB52F0CT4hWNm5ruadrLZG3Fyhr+GeE8MxEjiPaMT9o/HhaP7RB6/Jg5YQd/nf2CAZjDms2iIKhfMcjKS5nbqTvscPsTHg5V9fn3p6r31iJOaN5J0B/dBI3BvJtwzebZxuOUhLZHf03yiz3bW+yLFTCdpLdcD4WBWgSYwCNnXsYay5wWT8lGqNpFjUJZgiZxiN2DHnHxATsrsJfYThcNSOvknNybc9SR+dOJy0Y66kJT1snWxfHQla5SDgKi1DLlH0yezymvMPiTlYD45dJyN2qkRq47mv8T3POPhKQ1qeVrmEVkvbHi6dsvuoelyGjgFUu3cgE3fcYevQ2aaijogRmS/i7ipxjxOQewH2OE3XB7Jj4qDlPF731XZ94VxaTZxgsmFsbzMv+t1ULufL70W7kvaJn8w6JuZjvfi81eOj4eO9+LykZDdbt6YkxXP74qtMKj9RiadfcvkRC5G03BsIEHcMXUeJ2HTUPSQl1S6pw2Nb8HwU5VDO7Z2UGJJvfYY+cnIo5va5lBRzb30PMXo3a2fZXGOLJrlb0v0ut3JjME5nGt55abe8MEIWwrlHysn8OHfcbNUf8G4xSgzHPeD6lTsOAO2cgCh52V2VvcBZFBKvu69c2tRqyN4lhULCmF6KjnJbdY/dVcArBLRPSPMhMDkBZkB403sqIqJ9/ff3TdGAU1NsDBVArKr0s4gu1BX6z3+mtsLYAu/8stGxHPNZMSPF1PSFla6qq3M1Oy9jtZCTCZY30M7i9l8IN06d3WyQ3fjXUz3Hp1CIh4PgwyBWAcuHImArj41iC40BFwzTur3FypTUyKK5KFWara+7g7ne1Hv0N5oRG83CjQ7sONQEfABBQK+hNZ83Ye7lV0ziMOtDbWtVuMnUFRUyJhvpg0X1CxVfSH+bObHNLNjmwD5DDcDDAQK6DK31m3vFmJZm7ygTU22uFFn+Joqmwl5UCZ0Ym38W0dtUfezR5s+JrWYDrA7sNtQIfAhCQMehtd9W8H9ZrJwsSLlLWCVK42Zr+L3Majw9ZhYo/lbuE1vJelgZ2DGoTeZ9TA7oCLT2HrSkysyWRrKX1yrwlJhkre0BcAGdHD7Mnv2eLb9XU9VD5R4Q28yCbQ7sJNQAPBwgoMvQWo9bm3+rsYCFPdVXbV46FoGRpQ0uSvgRQ9/oMUtYpxXiqbfNh8Q2s2CbA7sMNQAPBwjoMrTWtzYu4Z8Uy+vS1DnJZSWVdjsfymbcu7I1C728/oQDwtQhnPdwQUfElrOBlgd2H2oMPhQjoBPRMhzfWlW7uH5c1SZ0Af8tr9G3gsFKpzAA/E09JjaV9TU1sJtQ28172x3QL0iNdlt5Z020qDYzn8l7K8VHaa/IqPVUFLYP4w2iud3Z87aYaj/H2csC7Q3rI8TG81Dj+3cUUsv3Wj5IZstvuLGC6hzzxiXMiJG4krqJC8Vp0SMSR7Vz4uxkPe0M7Ba0RvO+Rgd0B0qL2casAXbaGljQVZff3US3viU4FjPwlNqUy/M1lpEay0KMDewYtJbzIMsDegeF2f8PFkiC4A=='
SS3_BOOTSTRAP_BYTES = zlib.decompress(base64.b64decode(SS3_BOOTSTRAP_PAYLOAD))
BOOTSTRAP_DF=pd.read_csv(io.BytesIO(SS3_BOOTSTRAP_BYTES),dtype=str).fillna('')
assert len(BOOTSTRAP_DF)==252
assert not set(BOOTSTRAP_DF.loc[BOOTSTRAP_DF.partition=='train','group_id']) & set(BOOTSTRAP_DF.loc[BOOTSTRAP_DF.partition=='dev','group_id'])
print('Corpus base:',len(BOOTSTRAP_DF),'sha256:',hashlib.sha256(SS3_BOOTSTRAP_BYTES).hexdigest())


Corpus base: 252 sha256: 4d66b4060d7adbd7c771f8fd3849abafcaab28d926f9b72f31602a6242af8de7


In [20]:
# Ampliación semántica: más estructuras lingüísticas, ámbitos y negaciones.
# Plantillas destinadas EXCLUSIVAMENTE a train, nunca al test externo.
SS3_OPERATION_FRAMES={
'documental':[
'Localizá el apartado normativo sobre {topic} y precisá qué ordena.',
'Identificá los artículos concretos de {topic} y citá el alcance literal.',
'¿Qué sujetos contempla la disposición sobre {topic}? Explicá los requisitos.',
'Indicá cuáles son las atribuciones previstas en el decreto referido a {topic}.',
'¿Cuál es el contenido exacto de la regulación de {topic}?',
'Examiná qué especifica el texto legal acerca de {topic} sin atribuir efectos.',
'¿Qué obligaciones se desprenden del artículo asociado a {topic}?',
'Extraé las condiciones formales y destinatarios del régimen de {topic}.',
'¿Qué dice la fuente primaria sobre {topic} y dónde aparece?',
'Precisá el dispositivo normativo que establece {topic}.',
],
'comparative':[
'Examiná convergencias y diferencias entre provincias respecto de {topic}.',
'¿Hasta qué punto son equivalentes las intervenciones provinciales sobre {topic}?',
'Contrastá dos territorios usando la misma variable de {topic}.',
'Describí los mecanismos coincidentes y divergentes en {topic} de cada jurisdicción.',
'Compará las estrategias provinciales de {topic} sin mezclar las escalas.',
'Elaborá una matriz de semejanzas y divergencias institucionales de {topic}.',
'¿Qué provincia contempla {topic} por un instrumento comparable a las otras?',
'Situá los documentos de cada distrito frente a la misma categoría de {topic}.',
'Buscá similitudes verificables y diferencias relevantes de {topic} por provincia.',
'Confrontá los instrumentos de {topic} de distintas jurisdicciones.',
],
'diachronic':[
'Describí qué se transformó y qué persistió en {topic} a lo largo de una década.',
'Reconstruí las etapas iniciales y finales de {topic} sin suponer continuidad.',
'¿Cuáles fueron las rupturas y permanencias históricas en {topic}?',
'Compará los momentos tempranos, intermedios y tardíos de {topic}.',
'Identificá secuencias documentales y discontinuidades de {topic} durante el período.',
'¿Cómo se modificó la orientación institucional hacia {topic} en diez años?',
'Seguí la trayectoria temporal de {topic} con base en documentos fechados.',
'¿Qué instrumentos de {topic} se mantienen y cuáles desaparecen de la evidencia?',
'Reconstruí una periodización de {topic} sin equiparar fecha con implementación.',
'Contrastá comienzo, transición y cierre del período para {topic}.',
],
'hypothesis_test':[
'¿Qué documentos respaldan y cuáles limitan la lectura propuesta sobre {topic}?',
'Buscá casos que tensionen nuestra interpretación de {topic}, sin forzar refutación.',
'¿Hay contraejemplos capaces de matizar la hipótesis sobre {topic}?',
'Distingui pruebas favorables, heterogeneidad y contradicción en {topic}.',
'Probá la consistencia de una caracterización discursiva sobre {topic}.',
'Contrastá la tesis sobre {topic} con evidencia de orientaciones diferentes.',
'¿Qué hallazgos debilitan la hipótesis central sobre {topic}?',
'Someté a prueba la lectura de {topic} con fuentes de otra clase.',
'¿Hasta dónde resiste la caracterización de {topic} ante ejemplos opuestos?',
'Diferenciá evidencia adversarial de simples matices en {topic}.',
],
'coverage':[
'¿Qué parte de la documentación disponible de {topic} quedó incompleta?',
'¿Cuál es la cobertura y calidad de extracción del corpus de {topic}?',
'Auditá documentos sin texto utilizable en el repositorio de {topic}.',
'Identificá pérdidas de lectura y fallos OCR del material de {topic}.',
'¿Qué documentos del inventario de {topic} necesitaron OCR?',
'Contabilizá archivos procesados y fragmentos legibles asociados a {topic}.',
'¿Hay páginas ausentes o extracción defectuosa en el corpus de {topic}?',
'¿Qué tan completo está el inventario documental de {topic}?',
'Revisá el estado de procesamiento de las fuentes de {topic}.',
'Describí la integridad de los archivos de {topic} antes del análisis.',
],
'out_of_scope':[
'¿Cuál es la cotización de hoy de {topic}?',
'Buscá el precio en vivo y las ofertas de {topic}.',
'Necesito el pronóstico meteorológico para mañana en {topic}.',
'¿Qué novedades de esta mañana aparecen sobre {topic} en redes?',
'Consultá el valor de mercado de {topic} ahora mismo.',
'¿Quién ganó el partido de ayer de {topic}?',
'¿Qué locales venden hoy {topic} cerca de mí?',
'¿Cuánto cuesta comprar {topic} este fin de semana?',
'¿Cómo estuvo la cotización intradiaria de {topic}?',
'¿Podés encontrar disponibilidad de reservas para {topic} esta noche?',
],
'thematic':[
'Resumí los ejes discursivos presentes en los documentos sobre {topic}.',
'¿Qué temas predominan en los informes que tratan {topic}?',
'Caracterizá las principales narrativas que emergen en torno a {topic}.',
'Describí orientaciones institucionales relacionadas con {topic}.',
'¿Qué patrones argumentativos aparecen sobre {topic} en las fuentes?',
'Hacé una lectura temática de los documentos de {topic}.',
'¿Cómo se presenta discursivamente el desarrollo de {topic}?',
'Identificá el repertorio de asuntos más recurrentes alrededor de {topic}.',
'Sintetizá las distintas formas de representar {topic} en el corpus.',
'¿Qué enfoques documentales conviven en el campo de {topic}?',
]}
SS3_TOPICS=[('apoyo financiero a empresas','funding'),('financiación de la infraestructura','funding'),
('desarrollo de tecnologías científicas','technology'),('innovación de software','technology'),
('productividad de cadenas hortícolas','production'),('fomento agroindustrial','production'),
('campañas públicas y difusión oficial','communication'),('divulgación institucional','communication'),
('efectos sobre empleo e ingresos','impact'),('resultados sociales observados','impact'),
('autoridades y competencias administrativas','institutional'),('adopción de marcos regulatorios','institutional')]
SS3_ADDITIONAL=[]
for operation,patterns in SS3_OPERATION_FRAMES.items():
    for fi,frm in enumerate(patterns):
        for ti,(topic,dimension) in enumerate(SS3_TOPICS):
            # Sólo una parte de combinaciones para entrenamiento manejable y grupos separados.
            if ti % 3 != fi % 3: continue
            q=frm.format(topic=topic)
            risks='institutional_attribution' if operation=='documental' and fi%4==0 else ''
            if operation=='hypothesis_test' and fi%3==0: risks='negative_evidence'
            SS3_ADDITIONAL.append({'sample_id':f'AUG-{operation}-{fi:02d}-{ti:02d}',
              'question':q,'operation':operation,'risks':risks,'dimensions':dimension,
              'group_id':f'AUG-{operation}-{fi:02d}','partition':'train',
              'source':'SYNTHETIC_AUGMENTATION','label_review_status':'provisional_not_human_reviewed'})

# Los riesgos se entrenan con ejemplos contrastivos específicos, no se usan como motivos automáticos de FAST.
SS3_RISK_PHRASES={
'causal_effect':['¿Se puede atribuir causalmente el aumento del empleo al programa de {topic}?',
  '¿La política sobre {topic} produjo un resultado económico demostrable?',
  '¿Qué cambió por efecto de la intervención pública sobre {topic}?'],
'negative_evidence':['¿Que no encontremos papeles sobre {topic} prueba que nunca existió?',
  '¿Podemos concluir inexistencia histórica por no hallar registros de {topic}?',
  '¿Es válido afirmar ausencia absoluta de {topic} por silencio del archivo?'],
'superlative':['¿Cuál jurisdicción es la más avanzada en {topic}?',
  '¿Qué provincia tuvo mayor desarrollo relativo de {topic}?',
  '¿Podemos designar la región líder de {topic}?'],
'outcome_efficacy':['Ordená los programas de {topic} de mayor a menor eficacia.',
  '¿Cuál consiguió el mejor resultado social comprobado en {topic}?',
  'Clasificá la eficacia de las intervenciones en {topic} con impactos mensurables.'],
'institutional_attribution':['¿Qué responsabilidades atribuye la norma de {topic} a sus autoridades?',
  '¿Qué órgano es competente en el decreto sobre {topic}?',
  '¿A qué institución asigna facultades el instrumento de {topic}?']}
for risk,frames in SS3_RISK_PHRASES.items():
    for fi,fr in enumerate(frames):
        for ti,(topic,dimension) in enumerate(SS3_TOPICS):
            if ti%2!=fi%2:continue
            SS3_ADDITIONAL.append({'sample_id':f'RISK-{risk}-{fi}-{ti}','question':fr.format(topic=topic),
              'operation':'hypothesis_test' if risk in ('negative_evidence','causal_effect') else 'comparative' if risk in ('superlative','outcome_efficacy') else 'documental',
              'risks':risk,'dimensions':dimension,'group_id':f'RISK-{risk}-{fi}','partition':'train',
              'source':'SYNTHETIC_AUGMENTATION','label_review_status':'provisional_not_human_reviewed'})

SS3_AUGMENTED=pd.DataFrame(SS3_ADDITIONAL)
SS3_TRAIN_FRAME=pd.concat([BOOTSTRAP_DF.query("partition=='train'"),SS3_AUGMENTED],ignore_index=True)
SS3_DEV_FRAME=BOOTSTRAP_DF.query("partition=='dev'").copy()
SS3_TRAIN_FRAME=SS3_TRAIN_FRAME.drop_duplicates('question')
assert not set(SS3_TRAIN_FRAME.group_id)&set(SS3_DEV_FRAME.group_id)
assert not set(SS3_TRAIN_FRAME.question.str.lower())&set(SS3_DEV_FRAME.question.str.lower())
SS3_TRAIN_FRAME.to_csv(OUTPUT_DIR/'ss3_training_augmented_v320.csv',index=False,encoding='utf-8-sig')
SS3_DEV_FRAME.to_csv(OUTPUT_DIR/'ss3_dev_untouched_v320.csv',index=False,encoding='utf-8-sig')
print('SS3 train:',len(SS3_TRAIN_FRAME),'dev:',len(SS3_DEV_FRAME))


SS3 train: 559 dev: 63


## 8. Normalización antisesgo, PySS3 multieje y calibración de riesgos

In [21]:
from pyss3 import SS3
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from functools import lru_cache

# No se asigna ninguna operación por reglas: máscara para evitar atajos por provincia, año o nombres.
def ss3_training_view(text):
    t=normalize_text(str(text))
    for variants in PROVINCE_VARIANTS.values():
        for variant in sorted(variants,key=len,reverse=True):
            t=re.sub(r'(?<!\w)'+re.escape(normalize_text(variant))+r'(?!\w)',' territorio ',t)
    t=re.sub(r'\b(?:19|20)\d{2}\b',' periodo ',t)
    t=re.sub(r'\b(?:doc[_ -]?\d+|ley\s+\d+|decreto\s+\d+)\b',' documento ',t)
    return re.sub(r'\s+',' ',t).strip()

SS3_PARAMETERS={'s':0.25,'l':1.0,'p':2.0}
SS3_MAX_NGRAM=3
SS3_RISKS=['institutional_attribution','causal_effect','negative_evidence','superlative','outcome_efficacy']
SS3_CLASSIFIERS={}
SS3_RISK_CLASSIFIERS={}

training_x=[ss3_training_view(q) for q in SS3_TRAIN_FRAME.question]
for axis in ['operation','dimensions']:
    m=SS3(**SS3_PARAMETERS)
    m.fit(training_x,SS3_TRAIN_FRAME[axis].tolist(),n_grams=SS3_MAX_NGRAM,leave_pbar=False)
    SS3_CLASSIFIERS[axis]=m
for risk in SS3_RISKS:
    labels=['positive' if risk in str(x).split('|') else 'negative' for x in SS3_TRAIN_FRAME.risks]
    assert sum(x=='positive' for x in labels)>=8
    m=SS3(**SS3_PARAMETERS)
    m.fit(training_x,labels,n_grams=SS3_MAX_NGRAM,leave_pbar=False)
    SS3_RISK_CLASSIFIERS[risk]=m

# Hipótesis específica clasificada mediante un tercer SS3, no regex de operación.
HYPOTHESIS_ROWS=[]
for tag,expressions in {
 'modelo_comunicacional':['narrativa comunicacional','estrategia de comunicación','discurso institucional','campañas oficiales','estilo comunicacional','difusión política'],
 'modelo_modernizador':['modelo modernizador','ciencia e innovación','desarrollo tecnológico','innovación científica','modernización pública','fomento tecnológico'],
 'modelo_productivista':['modelo productivista','producción provincial','estrategia agroindustrial','política productiva','desarrollo de las cadenas','incentivos industriales'],
 'sin_hipotesis':['consultá fuentes','qué dispone la norma','resumí el documento','identificá artículos','extraé responsables','auditoría del corpus']
}.items():
    for i,e in enumerate(expressions):
        for prefix in ['Examiná la caracterización de','¿Qué evidencia hay sobre']:
            HYPOTHESIS_ROWS.append((ss3_training_view(prefix+' '+e+'?'),tag))
SS3_HYPOTHESIS=SS3(**SS3_PARAMETERS)
SS3_HYPOTHESIS.fit([q for q,_ in HYPOTHESIS_ROWS],[y for _,y in HYPOTHESIS_ROWS],n_grams=2,leave_pbar=False)

SS3_FOCUS_ROWS=[]
for label,frames in {
 'adversarial':['Buscá contraevidencia y ejemplos que contradigan','Qué documentos se oponen a la hipótesis','Quiero una prueba adversarial frente a','Hay casos que desmientan'],
 'qualifying':['Qué elementos obligan a matizar','Buscá excepciones y límites de','Hay evidencias favorables y matices sobre','Cómo podría calificarse la caracterización de']
}.items():
    for f in frames:
        for topic in ['modelo modernizador','agenda comunicacional','estrategia productivista','políticas de innovación']:
            SS3_FOCUS_ROWS.append((ss3_training_view(f+' '+topic),label))
SS3_EVIDENCE_FOCUS=SS3(**SS3_PARAMETERS)
SS3_EVIDENCE_FOCUS.fit([q for q,_ in SS3_FOCUS_ROWS],[y for _,y in SS3_FOCUS_ROWS],n_grams=2,leave_pbar=False)


# Selección de thresholds de riesgos por F1 en el conjunto de desarrollo sintético.
# El test externo NO participa en esta calibración.
def _scores(model,text):
    return {str(model.get_category_name(int(i))):float(s) for i,s in model.classify(text,sort=True)}

def _risk_margin(model,text):
    d=_scores(model,text)
    return float(d.get('positive',0)-d.get('negative',0))

RISK_THRESHOLDS={}
for risk,model in SS3_RISK_CLASSIFIERS.items():
    x=[ss3_training_view(q) for q in SS3_DEV_FRAME.question]
    y=np.array([int(risk in str(z).split('|')) for z in SS3_DEV_FRAME.risks],dtype=int)
    margins=np.array([_risk_margin(model,t) for t in x])
    # Un conjunto finito de umbrales, ajustados SOLAMENTE sobre dev, registrados.
    candidates=sorted(set([0.0]+list(np.quantile(margins,np.linspace(0.05,0.95,19)))))
    opts=[(f1_score(y,(margins>=t).astype(int),zero_division=0),-abs(t),t) for t in candidates]
    RISK_THRESHOLDS[risk]=float(max(opts)[2])

print('SS3 modelos entrenados:',len(SS3_CLASSIFIERS)+len(SS3_RISK_CLASSIFIERS)+2)
print('Umbrales de riesgo calibrados en dev:',RISK_THRESHOLDS)


SS3 modelos entrenados: 9
Umbrales de riesgo calibrados en dev: {'institutional_attribution': -0.5491004867761126, 'causal_effect': -1.8494945883325207, 'negative_evidence': 0.03543504311545809, 'superlative': -2.529179154547467, 'outcome_efficacy': -3.508825208306482}


## 9. Plan semántico único: SS3 decide, sin fallback al planner anterior

In [22]:
SS3_OPERATION_SET=set(SS3_CLASSIFIERS['operation'].get_categories())
SS3_TOPIC_DESCRIPTORS={
'funding':'financiamiento créditos fondos subsidios instrumentos fiscales asistencia financiera',
'production':'producción agroindustrial cadenas de valor competitividad productores',
'technology':'ciencia innovación tecnologías investigación emprendimientos software',
'communication':'comunicación institucional estrategias difusión imagen pública',
'impact':'resultados observados empleo impacto indicadores cumplimiento',
'institutional':'normativa leyes decretos competencias administración pública',
'other':'políticas provinciales documentos institucionales'}

@dataclass(frozen=True)
class SS3SemanticPlan:
    question: str
    operation: str
    secondary_operations: tuple[str,...]
    dimension: str
    risks: tuple[str,...]
    hypothesis: str | None
    evidence_focus: str
    provinces: tuple[str,...]
    periods: tuple[str,...]
    sources: tuple[str,...]
    retrieval_query: str
    route: str
    max_documents: int
    max_evidence: int
    max_new_tokens: int
    reasons: dict
    plan_hash: str

# Los scores no son probabilidades: sólo confianza relativa dentro del SS3.
def _ss3_axis(model,raw):
    text=ss3_training_view(raw)
    ranked=_scores(model,text)
    ordering=sorted(ranked.items(),key=lambda x:x[1],reverse=True)
    label=str(model.classify_label(text,def_cat='most-probable'))
    if label not in ranked or (ordering and label!=ordering[0][0]):
        label=ordering[0][0] if ordering else 'unclassified'
    terms=[]
    if label in model.get_categories():
        try: terms=[{'term':str(t),'cv':float(s)} for t,s in model.extract_insight(text,cat=label,level='word',window_size=0,min_cv=0.005,sort=True)[:8]]
        except Exception:pass
    return label,ordering,terms

def ss3_scope_literals(q,prior=None):
    # El alcance literal no tiene poder para recategorizar operaciones ni riesgos.
    provinces=detect_provinces(q)
    sources=detect_sources(q)
    # Detecta intervalo calendario completo, sin inferir operación diacrónica.
    norm=normalize_text(q)
    year_matches=[int(x) for x in re.findall(r'\b(?:2009|201[0-9])\b',norm)]
    periods=detect_periods(q)
    if len(year_matches)>=2 and max(year_matches)-min(year_matches)>=2:
        for y in range(min(year_matches),max(year_matches)+1):
            if y<=2011:p='2009–2011'
            elif y<=2015:p='2012–2015'
            else:p='2016–2019'
            if p not in periods: periods.append(p)
    if prior and len(norm.split())<=12:
        if not provinces:provinces=list(prior.get('provinces',[]))
        if not sources:sources=list(prior.get('sources',[]))
        if not periods:periods=list(prior.get('periods',[]))
    if not provinces: provinces=PROVINCES.copy()
    return tuple(provinces),tuple(sources),tuple(sorted(set(periods)))

def ss3_make_plan(q, prior=None):
    q=str(q).strip()
    if not q:raise ValueError('Pregunta vacía')
    # SS3 ve contexto textual en seguimientos: nunca hereda automáticamente la clase.
    classify_text=q
    if prior is not None and len(normalize_text(q).split())<=12:
        classify_text=(str(prior.get('question',''))+'\nContinuación: '+q)
    op,oranking,oterms=_ss3_axis(SS3_CLASSIFIERS['operation'],classify_text)
    dim,dranking,dterms=_ss3_axis(SS3_CLASSIFIERS['dimensions'],classify_text)
    hyp,hranking,hterms=_ss3_axis(SS3_HYPOTHESIS,classify_text)
    evidence_focus,eranking,eterms=_ss3_axis(SS3_EVIDENCE_FOCUS,classify_text)
    risks=[];risk_detail={}
    for risk,model in SS3_RISK_CLASSIFIERS.items():
        margin=_risk_margin(model,ss3_training_view(classify_text))
        risk_detail[risk]={'margin':margin,'threshold_dev':RISK_THRESHOLDS[risk],
                           'activated': bool(margin>=RISK_THRESHOLDS[risk])}
        if margin>=RISK_THRESHOLDS[risk]:risks.append(risk)
    sec=tuple(x for x,s in oranking[1:4] if op not in ['out_of_scope','coverage'] and s>0
         and s>=oranking[0][1]*0.72 and x not in ('out_of_scope','coverage'))
    provinces,sources,periods=ss3_scope_literals(q,prior)
    if op not in SS3_OPERATION_SET: raise RuntimeError('SS3 retornó operación inválida: '+op)
    if op=='out_of_scope':route='OUT_OF_SCOPE'
    elif op=='coverage':route='CORPUS_COVERAGE'
    else:route='RETRIEVAL_LLM'
    # El presupuesto es traducción mecánica de las clases de SS3, no un parser paralelo.
    budget={'documental':(5,7,640),'thematic':(8,12,850),'comparative':(10,12,1050),
            'diachronic':(10,12,1100),'hypothesis_test':(10,12,1100),
            'coverage':(0,0,0),'out_of_scope':(0,0,0)}
    maxdocs,maxevid,maxtok=budget[op]
    core=semantic_core(q,list(provinces),list(sources))
    # SS3 decide dimensión y operaciones que enriquecen recuperación.
    enrich=' '.join([core,SS3_TOPIC_DESCRIPTORS.get(dim,'')])
    if prior is not None and len(normalize_text(q).split())<=12:
        enrich=str(prior.get('retrieval_query',''))+' '+core+' '+SS3_TOPIC_DESCRIPTORS.get(dim,'')
    hy=None if hyp=='sin_hipotesis' else hyp
    if op!='hypothesis_test':hy=None
    rationale={'authority':'SS3_ONLY','classification_input':classify_text,
               'operation_scores':oranking,'operation_terms':oterms,
               'dimension_scores':dranking,'dimension_terms':dterms,
               'hypothesis_scores':hranking,'hypothesis_terms':hterms,
               'evidence_focus_scores':eranking,'evidence_focus_terms':eterms,
               'risk_scores':risk_detail,'fallback_used':False,
               'metadata_scope_parsing_is_not_semantic_planning':True}
    body=dict(question=q,operation=op,secondary_operations=sec,dimension=dim,
              risks=tuple(risks),hypothesis=hy,evidence_focus=evidence_focus,provinces=provinces,
              periods=periods,sources=sources,retrieval_query=enrich.strip(),
              route=route,max_documents=maxdocs,max_evidence=maxevid,max_new_tokens=maxtok,
              reasons=rationale)
    signature=hashlib.sha256(json.dumps(body,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()[:16]
    return SS3SemanticPlan(**body,plan_hash='SS3-'+signature)

def snapshot_from_ss3(plan):
    payload=dict(query=plan.question,query_type=plan.operation,semantic_core=plan.retrieval_query,
      provinces=plan.provinces,sources=plan.sources,periods=plan.periods,
      hypothesis=plan.hypothesis,expected_answer_mode='ss3_sole_planner',
      required_controls=tuple(['operation_'+plan.operation,'ss3_dimension_'+plan.dimension,
       'ss3_evidence_focus_'+plan.evidence_focus]
       +['ss3_secondary_'+o for o in plan.secondary_operations]
       +['ss3_risk_'+r for r in plan.risks]),runtime_signature=RUNTIME_SIGNATURE)
    return QueryIntentSnapshot(**payload,signature='QIS-'+plan.plan_hash)

assert 'live_snapshot' not in globals() and 'choose_execution_plan' not in globals()
assert 'infer_axes' not in globals() and 'BASELINE_EXECUTION_FN' not in globals()
print('PLANIFICADOR ÚNICO:', PLANNER_IMPLEMENTATION)


PLANIFICADOR ÚNICO: PYSS3_ONLY_V320


## 10. Medición interna SS3, sin usar las preguntas externas

In [23]:
def run_ss3_internal_dev():
    rows=[]
    for z in SS3_DEV_FRAME.to_dict('records'):
        p=ss3_make_plan(z['question'])
        rows.append({'question':z['question'],'expected_operation':z['operation'],'ss3_operation':p.operation,
                     'expected_dimension':z['dimensions'],'ss3_dimension':p.dimension,
                     'expected_risks':z['risks'],'ss3_risks':'|'.join(p.risks),
                     'plan_hash':p.plan_hash})
    data=pd.DataFrame(rows)
    result={'n':len(data),'operation_accuracy':accuracy_score(data.expected_operation,data.ss3_operation),
      'operation_f1_macro':f1_score(data.expected_operation,data.ss3_operation,average='macro',zero_division=0),
      'dimension_f1_macro':f1_score(data.expected_dimension,data.ss3_dimension,average='macro',zero_division=0)}
    truth=np.array([[int(r in str(v).split('|')) for r in SS3_RISKS] for v in data.expected_risks])
    pred=np.array([[int(r in str(v).split('|')) for r in SS3_RISKS] for v in data.ss3_risks])
    result['risk_f1_micro']=f1_score(truth.ravel(),pred.ravel(),zero_division=0)
    data.to_csv(OUTPUT_DIR/'ss3_internal_dev_predictions_v320.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame([result]).to_csv(OUTPUT_DIR/'ss3_internal_dev_metrics_v320.csv',index=False)
    return result,data
SS3_DEV_RESULT,SS3_DEV_PRED=run_ss3_internal_dev()
display(pd.DataFrame([SS3_DEV_RESULT]))


,n,operation_accuracy,operation_f1_macro,dimension_f1_macro,risk_f1_micro
0,63,0.52381,0.460204,0.975724,0.392857


## 11. Un solo flujo SS3 → dossier → Ministral: trazabilidad completa

In [24]:
import secrets, time
from time import perf_counter

SS3_AUDIT_PATH=OUTPUT_DIR/'ss3_only_full_audit_v320.jsonl'
SS3_RUNTIME_TURNS=[]
SS3_SYSTEM_PROMPT=NARRATIVE_SYSTEM_PROMPT+"""
\nINTEGRACIÓN DIRECTA SS3 V3.2.0:
- La operación, las dimensiones, riesgos y alternativas vienen SOLO de SS3; no vuelvas a reclasificar la intención.
- El plan SS3 es autoritativo para decidir CÓMO responder; no es una fuente de hechos.
- Distinguir normas, propuestas, ejecución y resultados comprobados.
- Etiquetas adversarial y qualifying NO implican por sí solas refutación de una hipótesis.
- En superlativos, rankings y efectos causales, declarar ausencia de indicadores comparables si el dossier no los acredita.
- La ausencia de un fragmento nunca demuestra inexistencia histórica.
- Comparaciones deben tratar la MISMA variable por territorio y período; si no hay equivalencias, indicarlo.
- Las respuestas deben atender todas las dimensiones solicitadas con citas [E001] y cerrar en <FIN>.
"""

def ss3_llm_generate(dossier, contract, plan):
    meta={'model_id':MINISTRAL_MODEL_ID,'ready':bool(LLM_RUNTIME.get('ready')),
          'generation_complete':False,'input_tokens':0,'generated_tokens':0,'error':''}
    if not LLM_RUNTIME.get('ready'):
        meta['error']=LLM_RUNTIME.get('error','Ministral no cargado');return '',meta
    payload=dossier_for_prompt(dossier,contract)
    payload['ss3_authoritative_semantic_plan']={
      'operation':plan.operation,'secondary_operations':list(plan.secondary_operations),
      'dimension':plan.dimension,'inference_risks':list(plan.risks),
      'hypothesis':plan.hypothesis,'evidence_focus':plan.evidence_focus,'scope':{'provinces':plan.provinces,
       'sources':plan.sources,'periods':plan.periods},
      'explanations':plan.reasons,'factual_authority':False}
    for e in payload['evidence']:
        e['text']=str(e.get('text',''))[:1100]
        e['context']=str(e.get('context',''))[:150]
    msgs=[{'role':'system','content':SS3_SYSTEM_PROMPT},
          {'role':'user','content':json.dumps(payload,ensure_ascii=False,default=str)}]
    start=perf_counter()
    try:
        inputs=EXPLAINER_TOKENIZER.apply_chat_template(msgs,add_generation_prompt=True,
           return_tensors='pt',return_dict=True)
        dev=next(EXPLAINER_MODEL.parameters()).device
        inputs={k:(v.to(dev) if hasattr(v,'to') else v) for k,v in inputs.items()}
        n=int(inputs['input_ids'].shape[-1])
        with torch.inference_mode():
            gen=EXPLAINER_MODEL.generate(**inputs,max_new_tokens=plan.max_new_tokens,
                    do_sample=False,use_cache=True)[0]
        decoded=EXPLAINER_TOKENIZER.decode(gen[n:],skip_special_tokens=True).strip()
        meta.update({'input_tokens':n,'generated_tokens':int(gen.shape[-1]-n),
            'generation_complete':decoded.endswith('<FIN>'),'latency_seconds':perf_counter()-start})
        return decoded,meta
    except Exception as ex:
        meta.update({'error':type(ex).__name__+': '+str(ex),'latency_seconds':perf_counter()-start})
        return '',meta

def ss3_run_question(question,previous=None,with_llm=True,case_id=None):
    start=perf_counter();p=ss3_make_plan(question,previous)
    snap=snapshot_from_ss3(p)
    ident=case_id or ('S3-'+p.plan_hash)
    record={'case_id':ident,'question':question,'plan':asdict(p),'snapshot':asdict(snap),
       'status':'','documents':[],'evidence':[],'contract':{},'llm':{},
       'raw_answer':'','answer':'','validation':[],'ms_total':None,'version':VERSION}
    if p.route=='OUT_OF_SCOPE':
        record.update(status='SS3_OUT_OF_SCOPE',answer='SS3 clasificó la consulta fuera del alcance histórico-documental del Observatorio. No hay respuesta documental aplicable.')
    elif p.route=='CORPUS_COVERAGE':
        d=coverage_dossier(ident,snap)
        record['contract']=d.composition_contract
        record['coverage']=d.coverage_audit
        record.update(status='SS3_COVERAGE',answer=(f"Auditoría del corpus: {d.coverage_audit['documents']} documentos; "
           f"{d.coverage_audit['complete_documents']} con texto utilizable; "
           f"{d.coverage_audit['incomplete_documents']} con extracción incompleta. "
           'Esta cobertura no equivale a exhaustividad histórica.'))
    else:
        # Ningún Gold literal ni planificador determinístico interviene.
        try:
            docs=dynamic_document_candidates(snap,limit=p.max_documents)
            record['candidate_documents']=docs['doc_id'].astype(str).tolist() if not docs.empty else []
            if docs.empty: record.update(status='NO_DOCUMENTS',answer='No se localizaron documentos dentro del alcance solicitado.')
            else:
                ident='D-'+p.plan_hash
                intent_snapshots[ident]=snap
                d=dossier_from_documents(ident,snap,docs,limit=p.max_evidence)
                record['documents']=sorted(set(x.doc_id for x in d.evidence))
                record['evidence']=[asdict(x) for x in d.evidence]
                record['contract']=d.composition_contract
                if not d.composition_contract.get('passed'):
                    record.update(status='BLOCKED_EVIDENCE',answer='El dossier no satisface las obligaciones documentales del plan SS3. No se genera una conclusión.')
                elif not with_llm:
                    record['status']='DOSSIER_READY_NO_LLM'
                else:
                    ac=build_answer_contract(d)
                    raw,meta=ss3_llm_generate(d,ac,p)
                    record['raw_answer']=raw
                    record['llm']=meta
                    if not raw:record.update(status='LLM_UNAVAILABLE',answer='No se pudo ejecutar Ministral; no se simula la respuesta.')
                    else:
                        raw=re.sub(r'(?<!\w)\(\s*(E\d{3}(?:\s*,\s*E\d{3})*)\s*\)',lambda m:'['+m.group(1).replace(' ','')+']',raw)
                        v=validate_narrative(raw,d,ac)
                        record['validation']=v.to_dict('records')
                        accepted=accepted_text(v)
                        record['status']='GENERATED_COMPLETE' if meta['generation_complete'] else 'GENERATED_INCOMPLETE'
                        if not accepted.strip():record['status']='NO_VERIFIABLE_CLAIMS'
                        if not meta['generation_complete']:
                            record['answer']='**Borrador incompleto — no apto para publicación.**\n\n'+render_citations(accepted,d)
                        else:record['answer']=render_citations(accepted,d)
        except Exception as ex:
            record.update(status='ERROR',error=type(ex).__name__+': '+str(ex),answer='Error registrado: '+str(ex))
    record['ms_total']=round((perf_counter()-start)*1000,1)
    with SS3_AUDIT_PATH.open('a',encoding='utf-8') as f:
        f.write(json.dumps(record,ensure_ascii=False,default=str)+'\n')
    SS3_RUNTIME_TURNS.append(record)
    return record


## 12. Banco externo de preguntas completamente nuevas y regresión de 12 casos

In [25]:
# Los archivos de pruebas están INCLUIDOS dentro de esta notebook; ninguna pregunta se utiliza en fit().
SS3_TEST_80_PAYLOAD = 'case_id,family,question,expected_operation,expected_risks,expected_dimension,sequence_id,follow_up,label_status\nDOC01,DOC,Identificá qué órgano ejerce las competencias normativas de protección industrial en La Rioja.,documental,,institutional,DOC01,no,provisional_manual_review_required\nDOC02,DOC,¿Qué alcance exacto tiene la disposición sobre góndolas de productos riojanos?,documental,,institutional,DOC02,no,provisional_manual_review_required\nDOC03,DOC,Localizá el artículo que atribuye potestades presupuestarias en Catamarca Crece.,documental,,institutional,DOC03,no,provisional_manual_review_required\nDOC04,DOC,Precisá si la ley catamarqueña de innovación incluye asignación de recursos o sólo adhesión.,documental,,institutional,DOC04,no,provisional_manual_review_required\nDOC05,DOC,Describí qué exige la norma santiagueña sobre permisos comerciales cinegéticos.,documental,,institutional,DOC05,no,provisional_manual_review_required\nDOC06,DOC,¿Dónde fija la legislación riojana las acciones de apoyo al sector TIC?,documental,,institutional,DOC06,no,provisional_manual_review_required\nDOC07,DOC,Enumerá los sujetos afectados por las cláusulas de promoción comercial catamarqueña.,documental,,institutional,DOC07,no,provisional_manual_review_required\nDOC08,DOC,¿Qué establece formalmente el instrumento provincial para el financiamiento de pymes?,documental,,institutional,DOC08,no,provisional_manual_review_required\nDOC09,DOC,Transcribí de manera acotada los requisitos institucionales del régimen productivo.,documental,,institutional,DOC09,no,provisional_manual_review_required\nDOC10,DOC,Explicá qué función normativa se asigna a la autoridad de aplicación provincial.,documental,,institutional,DOC10,no,provisional_manual_review_required\nCOMP01,COMP,Confrontá la oferta de instrumentos crediticios para pequeños productores de Catamarca y La Rioja.,comparative,,funding,COMP01,no,provisional_manual_review_required\nCOMP02,COMP,¿Hay mecanismos institucionalmente semejantes en los programas agrícolas de dos provincias?,comparative,,funding,COMP02,no,provisional_manual_review_required\nCOMP03,COMP,Examiná si las normas de fomento empresarial tienen objetos comparables entre distritos.,comparative,,funding,COMP03,no,provisional_manual_review_required\nCOMP04,COMP,¿Dónde se parecen y se separan las políticas de apoyo económico de Catamarca y Santiago?,comparative,,funding,COMP04,no,provisional_manual_review_required\nCOMP05,COMP,Construí una comparación de beneficiarios potenciales sin equiparar montos ausentes.,comparative,,funding,COMP05,no,provisional_manual_review_required\nCOMP06,COMP,Identificá diferencias entre incentivos fiscales riojanos y financiamiento catamarqueño.,comparative,,funding,COMP06,no,provisional_manual_review_required\nCOMP07,COMP,Compará la capacitación como instrumento productivo en Santiago del Estero y La Rioja.,comparative,,funding,COMP07,no,provisional_manual_review_required\nCOMP08,COMP,Confrontá normas de ciencia con programas de divulgación en las tres provincias.,comparative,,funding,COMP08,no,provisional_manual_review_required\nCOMP09,COMP,¿Qué disposiciones ofrecen beneficios equivalentes y cuáles atienden actividades distintas?,comparative,,funding,COMP09,no,provisional_manual_review_required\nCOMP10,COMP,Contrastá normas y diagnósticos para no confundir dos tipos de instrumentos provinciales.,comparative,,funding,COMP10,no,provisional_manual_review_required\nTIME01,TIME,Rastreá las transformaciones en actividades científicas santiagueñas desde los inicios hasta el cierre del período.,diachronic,,technology,TIME01,no,provisional_manual_review_required\nTIME02,TIME,Diferenciá permanencias y rupturas de la formación tecnológica provincial en la década estudiada.,diachronic,,technology,TIME02,no,provisional_manual_review_required\nTIME03,TIME,¿Cómo cambió el énfasis en propiedad intelectual entre las etapas del Observatorio?,diachronic,,technology,TIME03,no,provisional_manual_review_required\nTIME04,TIME,Reconstruí secuencias de promoción científica sin asumir ejecución continua.,diachronic,,technology,TIME04,no,provisional_manual_review_required\nTIME05,TIME,¿Cuáles eran los dispositivos tecnológicos tempranos y cuáles aparecen en la fase final?,diachronic,,technology,TIME05,no,provisional_manual_review_required\nTIME06,TIME,Describí si la agenda del software se registra al principio o al final del estudio.,diachronic,,technology,TIME06,no,provisional_manual_review_required\nTIME07,TIME,Establecé una periodización de acciones de innovación en Santiago del Estero.,diachronic,,technology,TIME07,no,provisional_manual_review_required\nTIME08,TIME,¿Qué instrumentos productivos aparecen tempranamente y cuáles sólo en las fuentes tardías?,diachronic,,technology,TIME08,no,provisional_manual_review_required\nTIME09,TIME,Seguimiento histórico de difusión tecnológica en tres momentos de 2009 a 2019.,diachronic,,technology,TIME09,no,provisional_manual_review_required\nTIME10,TIME,Identificá sustituciones o persistencias institucionales en innovación a lo largo del decenio.,diachronic,,technology,TIME10,no,provisional_manual_review_required\nHYP01,HYP,¿Qué hallazgos matizan la imagen de Catamarca como provincia primordialmente comunicacional?,hypothesis_test,,communication,HYP01,no,provisional_manual_review_required\nHYP02,HYP,Encontrá orientaciones distintas de las campañas públicas para contrastar la caracterización catamarqueña.,hypothesis_test,,communication,HYP02,no,provisional_manual_review_required\nHYP03,HYP,¿Cuáles documentos limitan tratar el enfoque productivo como exclusivo en La Rioja?,hypothesis_test,,communication,HYP03,no,provisional_manual_review_required\nHYP04,HYP,Someté a prueba la idea de modernización santiagueña con fuentes sectoriales heterogéneas.,hypothesis_test,,communication,HYP04,no,provisional_manual_review_required\nHYP05,HYP,¿Puede refutarse la lectura comunicacional simplemente por encontrar una ley industrial?,hypothesis_test,,communication,HYP05,no,provisional_manual_review_required\nHYP06,HYP,Diferenciá contraejemplos reales y mera diversidad administrativa en las políticas provinciales.,hypothesis_test,,communication,HYP06,no,provisional_manual_review_required\nHYP07,HYP,¿La presencia de normas cinegéticas contradice una orientación hacia la innovación?,hypothesis_test,,communication,HYP07,no,provisional_manual_review_required\nHYP08,HYP,Discutí cuáles documentos obligan a reformular la interpretación del perfil riojano.,hypothesis_test,,communication,HYP08,no,provisional_manual_review_required\nHYP09,HYP,Evalúa la hipótesis de primacía tecnológica identificando explicaciones alternativas.,hypothesis_test,,communication,HYP09,no,provisional_manual_review_required\nHYP10,HYP,¿Qué hallazgo documental realmente incompatible pondría en duda el modelo modernizador?,hypothesis_test,,communication,HYP10,no,provisional_manual_review_required\nSAFETY01,SAFETY,¿Puede decirse que una ley provincial causó aumento de empleo sin series comparables?,hypothesis_test,causal_effect,impact,SAFETY01,no,provisional_manual_review_required\nSAFETY02,SAFETY,¿La falta de documentación prueba que una medida jamás existió en Catamarca?,hypothesis_test,negative_evidence,impact,SAFETY02,no,provisional_manual_review_required\nSAFETY03,SAFETY,¿Cuál modelo fue el más exitoso y cómo se comprobaría con indicadores homogéneos?,hypothesis_test,outcome_efficacy|superlative,impact,SAFETY03,no,provisional_manual_review_required\nSAFETY04,SAFETY,¿Podemos concluir que no ocurrió implementación porque sólo hay normas?,hypothesis_test,negative_evidence,impact,SAFETY04,no,provisional_manual_review_required\nSAFETY05,SAFETY,¿Qué límites tiene atribuir ingresos crecientes directamente a un programa?,hypothesis_test,causal_effect,impact,SAFETY05,no,provisional_manual_review_required\nSAFETY06,SAFETY,¿Se puede ordenar a las provincias por efectividad sin datos de resultados?,hypothesis_test,outcome_efficacy|superlative,impact,SAFETY06,no,provisional_manual_review_required\nSAFETY07,SAFETY,¿La mera aprobación de un decreto certifica que se alcanzaron sus metas?,hypothesis_test,causal_effect,impact,SAFETY07,no,provisional_manual_review_required\nSAFETY08,SAFETY,¿Qué evidencia adicional exigirías para acreditar causalidad en programas de empleo?,hypothesis_test,causal_effect,impact,SAFETY08,no,provisional_manual_review_required\nSAFETY09,SAFETY,¿Tiene sentido hablar de impacto social probado si sólo se registran objetivos?,hypothesis_test,causal_effect,impact,SAFETY09,no,provisional_manual_review_required\nSAFETY10,SAFETY,¿Qué diferencia hay entre un resultado que se anuncia y otro que se verifica?,hypothesis_test,,impact,SAFETY10,no,provisional_manual_review_required\nOOD01,OOD,¿Cuánto cuesta hoy el kilo de papa en un supermercado de Chilecito?,out_of_scope,,other,OOD01,no,provisional_manual_review_required\nOOD02,OOD,Consultá la temperatura prevista para mañana en Santiago del Estero.,out_of_scope,,other,OOD02,no,provisional_manual_review_required\nOOD03,OOD,¿Dónde puedo reservar una habitación esta noche en Catamarca capital?,out_of_scope,,other,OOD03,no,provisional_manual_review_required\nOOD04,OOD,¿Cuál es el precio actual del dólar financiero en la Argentina?,out_of_scope,,other,OOD04,no,provisional_manual_review_required\nOOD05,OOD,Decime qué ocurrió ayer en la bolsa de valores de Buenos Aires.,out_of_scope,,other,OOD05,no,provisional_manual_review_required\nOOD06,OOD,¿A qué hora juega hoy la selección nacional de fútbol?,out_of_scope,,other,OOD06,no,provisional_manual_review_required\nOOD07,OOD,Necesito la cotización de hoy del maíz por tonelada en el mercado.,out_of_scope,,other,OOD07,no,provisional_manual_review_required\nOOD08,OOD,¿Cuántos grados hace en este instante en la ciudad de La Rioja?,out_of_scope,,other,OOD08,no,provisional_manual_review_required\nOOD09,OOD,¿Qué farmacia está de turno ahora cerca de mi ubicación?,out_of_scope,,other,OOD09,no,provisional_manual_review_required\nOOD10,OOD,Buscá vuelos baratos disponibles este fin de semana para viajar al NOA.,out_of_scope,,other,OOD10,no,provisional_manual_review_required\nSYSTEM01,SYSTEM,¿Cuáles de los textos del corpus necesitaron procesamiento OCR para ser consultables?,coverage,,other,SYSTEM01,no,provisional_manual_review_required\nSYSTEM02,SYSTEM,Auditá el número de archivos que tienen texto utilizable en el índice cerrado.,coverage,,other,SYSTEM02,no,provisional_manual_review_required\nSYSTEM03,SYSTEM,¿Qué efectos sobre la recuperación podrían producir extracciones incompletas?,coverage,,other,SYSTEM03,no,provisional_manual_review_required\nSYSTEM04,SYSTEM,¿Se registran fallas de integridad de los documentos históricos disponibles?,coverage,,other,SYSTEM04,no,provisional_manual_review_required\nSYSTEM05,SYSTEM,¿Cómo describirías la cobertura documental y la ausencia de fuentes externas?,coverage,,other,SYSTEM05,no,provisional_manual_review_required\nSYSTEM06,SYSTEM,¿Qué partes del repositorio fueron inspeccionadas al cerrarse Retrieval?,coverage,,other,SYSTEM06,no,provisional_manual_review_required\nSYSTEM07,SYSTEM,Comprobá si todos los documentos indexados disponen de texto legible.,coverage,,other,SYSTEM07,no,provisional_manual_review_required\nSYSTEM08,SYSTEM,¿Cuál es el límite de exhaustividad del acervo digital disponible?,coverage,,other,SYSTEM08,no,provisional_manual_review_required\nSYSTEM09,SYSTEM,¿Cuántos documentos institucionales conforman el inventario de consulta?,coverage,,other,SYSTEM09,no,provisional_manual_review_required\nSYSTEM10,SYSTEM,¿Qué diferencia hay entre documentos no digitalizados y archivos incompletos?,coverage,,other,SYSTEM10,no,provisional_manual_review_required\nFOLLOW01-1,FOLLOW,¿Qué documentos respaldan o matizan el perfil modernizador de Santiago del Estero?,hypothesis_test,,technology,FUP01,no,provisional_manual_review_required\nFOLLOW01-2,FOLLOW,¿Y en Catamarca?,hypothesis_test,,technology,FUP01,yes,provisional_manual_review_required\nFOLLOW02-1,FOLLOW,Contrastá la capacitación productiva de La Rioja y Santiago del Estero.,comparative,,production,FUP02,no,provisional_manual_review_required\nFOLLOW02-2,FOLLOW,¿Y sólo con fuentes SAIJ?,comparative,,production,FUP02,yes,provisional_manual_review_required\nFOLLOW03-1,FOLLOW,Reconstruí la evolución del software y las TIC en La Rioja entre 2009 y 2019.,diachronic,,technology,FUP03,no,provisional_manual_review_required\nFOLLOW03-2,FOLLOW,¿Y en Catamarca?,diachronic,,technology,FUP03,yes,provisional_manual_review_required\nFOLLOW04-1,FOLLOW,¿Qué dispone el decreto que crea Catamarca Crece?,documental,,funding,FUP04,no,provisional_manual_review_required\nFOLLOW04-2,FOLLOW,¿Y quiénes pueden recibir los beneficios?,documental,,funding,FUP04,yes,provisional_manual_review_required\nFOLLOW05-1,FOLLOW,¿La falta de textos específicos prueba la inexistencia de programas tecnológicos riojanos?,hypothesis_test,negative_evidence,technology,FUP05,no,provisional_manual_review_required\nFOLLOW05-2,FOLLOW,¿Y sobre Santiago del Estero?,hypothesis_test,negative_evidence,technology,FUP05,yes,provisional_manual_review_required\n'
SS3_TEST_80 = pd.read_csv(io.StringIO(SS3_TEST_80_PAYLOAD),dtype=str).fillna('')
assert len(SS3_TEST_80)==80
assert not set(SS3_TEST_80.question.map(normalize_text)) & set(SS3_TRAIN_FRAME.question.map(normalize_text))
SS3_TEST_80.to_csv(OUTPUT_DIR/'ss3_external_80_v320.csv',index=False,encoding='utf-8-sig')
print('Test externo 80 casos, sin intersección textual con train:',len(SS3_TEST_80))


Test externo 80 casos, sin intersección textual con train: 80


## 13. Pruebas de planificación, dossiers y generación real de Ministral

In [26]:
def ss3_evaluate_80_prellm():
    rows=[]; prev={}
    for z in SS3_TEST_80.to_dict('records'):
        prevctx=prev.get(z['sequence_id']) if z.get('follow_up')=='yes' else None
        r=ss3_run_question(z['question'],previous=prevctx,with_llm=False,case_id=z['case_id'])
        p=r['plan']
        rows.append({'case_id':z['case_id'],'family':z['family'],'question':z['question'],
                     'expected_operation_preliminary':z['expected_operation'],
                     'operation_ss3':p['operation'],'operation_match':p['operation']==z['expected_operation'],
                     'expected_risks_preliminary':z['expected_risks'],'risks_ss3':'|'.join(p['risks']),
                     'expected_dimension_preliminary':z['expected_dimension'],'dimension_ss3':p['dimension'],
                     'route':p['route'],'status':r['status'],
                     'dossier_pass':r['contract'].get('passed') if r['contract'] else None,
                     'documents':'|'.join(r['documents']),
                     'operation_terms_json':json.dumps(p['reasons']['operation_terms'],ensure_ascii=False),
                     'risk_scores_json':json.dumps(p['reasons']['risk_scores'],ensure_ascii=False),
                     'plan_hash':p['plan_hash'],'ms_total':r['ms_total']})
        prev[z['sequence_id']]={'question':z['question'],'provinces':p['provinces'],
         'sources':p['sources'],'periods':p['periods'],'retrieval_query':p['retrieval_query']}
    data=pd.DataFrame(rows)
    data.to_csv(OUTPUT_DIR/'ss3_authority_80_plans_dossiers_v320.csv',index=False,encoding='utf-8-sig')
    overall={'n':len(data),'operation_accuracy_preliminary':float(data.operation_match.mean()),
             'operation_f1_macro_preliminary':float(f1_score(data.expected_operation_preliminary,
                data.operation_ss3,average='macro',zero_division=0)),
             'dossier_pass_rate_when_applicable':float(data.dossier_pass.dropna().astype(bool).mean())
               if data.dossier_pass.notna().any() else None,
             'out_of_scope_count':int((data.route=='OUT_OF_SCOPE').sum()),
             'errors':int((data.status=='ERROR').sum()),
             'human_verification':'PENDING'}
    pd.DataFrame([overall]).to_csv(OUTPUT_DIR/'ss3_authority_80_metrics_v320.csv',index=False)
    return data,overall

# Corrida pre-LLM con 80 casos, ejecución real de SS3 y RAG congelado.
RUN_80_PREFLIGHT=True
if RUN_80_PREFLIGHT:
    SS3_EXTERNAL_RESULTS, SS3_EXTERNAL_SUMMARY = ss3_evaluate_80_prellm()
    display(pd.DataFrame([SS3_EXTERNAL_SUMMARY]))
    display(pd.crosstab(SS3_EXTERNAL_RESULTS.expected_operation_preliminary,
                        SS3_EXTERNAL_RESULTS.operation_ss3))

# Tanda obligatoria por decisión del investigador: usa Ministral REAL, sin generación ficticia.
def ss3_generate_cases(case_ids,limit=None):
    if not LLM_RUNTIME.get('ready'): raise RuntimeError('Ministral no disponible: se necesita GPU e iniciar el modelo real.')
    items=SS3_TEST_80.set_index('case_id')
    todo=list(case_ids)
    if limit is not None:todo=todo[:int(limit)]
    results=[];history={}
    for cid in todo:
        if cid not in items.index:raise KeyError(cid)
        z=items.loc[cid]
        ctx=history.get(z['sequence_id']) if z['follow_up']=='yes' else None
        res=ss3_run_question(z['question'],previous=ctx,with_llm=True,case_id=cid)
        results.append({'case_id':cid,'question':z['question'],'operation':res['plan']['operation'],
          'risks':'|'.join(res['plan']['risks']),'status':res['status'],'answer':res['answer'],
          'n_evidence':len(res['evidence']),'n_claims_accepted':sum(1 for v in res['validation'] if v.get('status')=='ACCEPT'),
          'n_claims_rejected':sum(1 for v in res['validation'] if v.get('status')=='REJECT'),
          'generation_complete':res['llm'].get('generation_complete'),'ms_total':res['ms_total'],
          'audit_plan_hash':res['plan']['plan_hash']})
        history[z['sequence_id']]={'question':z['question'],'provinces':res['plan']['provinces'],
         'sources':res['plan']['sources'],'periods':res['plan']['periods'],
         'retrieval_query':res['plan']['retrieval_query']}
    frame=pd.DataFrame(results)
    frame.to_csv(OUTPUT_DIR/'ss3_ministral_e2e_v320.csv',index=False,encoding='utf-8-sig')
    return frame

# Opcional: RUN_E2E=True ejecutará 80 consultas en secuencia.
RUN_E2E=False
if RUN_E2E:
    display(ss3_generate_cases(SS3_TEST_80.case_id.tolist()))


,n,operation_accuracy_preliminary,operation_f1_macro_preliminary,dossier_pass_rate_when_applicable,out_of_scope_count,errors,human_verification
0,80,0.775,0.688312,0.985714,10,0,PENDING


operation_ss3,comparative,coverage,diachronic,documental,hypothesis_test,out_of_scope,thematic
expected_operation_preliminary,,,,,,,
comparative,10,1,0,0,1,0,0
coverage,0,9,0,0,1,0,0
diachronic,0,1,9,1,1,0,0
documental,0,0,0,11,1,0,0
hypothesis_test,2,4,1,1,14,1,1
out_of_scope,0,0,0,0,0,9,1


## 14. Criterio de validación falsable y revisión humana

In [27]:
SS3_REVIEW_80=SS3_TEST_80[['case_id','question','expected_operation','expected_risks','expected_dimension']].copy()
for field in ['human_review_operation','human_review_dimension','human_review_risks',
              'dossier_relevance_0_2','narrative_factually_grounded_0_2',
              'norm_vs_implementation_correct','conclusion_justified','observations']:
    SS3_REVIEW_80[field]=''
SS3_REVIEW_80.to_csv(OUTPUT_DIR/'ss3_human_review_80_v320.csv',index=False,encoding='utf-8-sig')

VALIDATION_CRITERIA={
 'operation_macro_f1':'>=0.75 en test externo tras revisión humana, no entrenado',
 'critical_abstention_false_positive':'<=2/80 preguntas, revisado manualmente',
 'critical_guardrail_regressions':'0 causales o rankings sin evidencia comparable',
 'dossier_structural_pass_rate':'>=0.90 entre rutas con dossier',
 'llm_generation_complete_rate':'>=0.90 entre consultas generadas',
 'substantive_review':'evaluación humana documentada sobre la pertinencia del dossier y las afirmaciones',
 'necessary_before_claiming_conclusive':'80 planes + dossiers + 80 generaciones GPU + etiquetas revisadas + auditoría de claims'}
(OUTPUT_DIR/'ss3_validation_preregistration_v320.json').write_text(
 json.dumps(VALIDATION_CRITERIA,ensure_ascii=False,indent=2),encoding='utf-8')
print('Criterios explícitos de evaluación:\n'+json.dumps(VALIDATION_CRITERIA,ensure_ascii=False,indent=2))


Criterios explícitos de evaluación:
{
  "operation_macro_f1": ">=0.75 en test externo tras revisión humana, no entrenado",
  "critical_abstention_false_positive": "<=2/80 preguntas, revisado manualmente",
  "critical_guardrail_regressions": "0 causales o rankings sin evidencia comparable",
  "dossier_structural_pass_rate": ">=0.90 entre rutas con dossier",
  "llm_generation_complete_rate": ">=0.90 entre consultas generadas",
  "substantive_review": "evaluación humana documentada sobre la pertinencia del dossier y las afirmaciones",
  "necessary_before_claiming_conclusive": "80 planes + dossiers + 80 generaciones GPU + etiquetas revisadas + auditoría de claims"
}


## 14.1. Dictamen reproducible de validación: no aprobar sin ejecución y revisión

In [28]:
def ss3_validation_verdict(review_path=None):
    """Evalúa la arquitectura SS3 única. No infiere validación científica de métricas no revisadas."""
    plan_path=OUTPUT_DIR/'ss3_authority_80_plans_dossiers_v320.csv'
    e2e_path=OUTPUT_DIR/'ss3_ministral_e2e_v320.csv'
    rv_path=Path(review_path) if review_path else OUTPUT_DIR/'ss3_human_review_80_v320.csv'
    if not plan_path.exists():
        return {'verdict':'NO_EVALUABLE','reason':'No hay preflight real de 80 planes.'}
    plans=pd.read_csv(plan_path,dtype=str).fillna('')
    if len(plans)!=80:
        return {'verdict':'NO_EVALUABLE','reason':f'Preflight incompleto: {len(plans)}/80.'}
    if not e2e_path.exists():
        return {'verdict':'PENDIENTE_GENERACION','reason':'Faltan las 80 respuestas de Ministral reales.'}
    generations=pd.read_csv(e2e_path,dtype=str).fillna('')
    if len(generations)!=80 or generations.case_id.nunique()!=80:
        return {'verdict':'PENDIENTE_GENERACION','reason':f'Resultados GPU incompletos: {len(generations)}/80.'}
    if not rv_path.exists():
        return {'verdict':'PENDIENTE_REVISION','reason':'Falta el CSV anotado por expertos.'}
    review=pd.read_csv(rv_path,dtype=str).fillna('')
    required=['human_review_operation','dossier_relevance_0_2','narrative_factually_grounded_0_2','norm_vs_implementation_correct','conclusion_justified']
    if len(review)!=80 or any(c not in review for c in required):
        return {'verdict':'PENDIENTE_REVISION','reason':'Plantilla incompleta o con campos requeridos ausentes.'}
    if any((review[c].str.strip()=='').any() for c in required):
        return {'verdict':'PENDIENTE_REVISION','reason':'Completar todas las anotaciones humanas en 80 casos.'}
    joined=plans.merge(review[['case_id','human_review_operation','dossier_relevance_0_2',
       'narrative_factually_grounded_0_2','norm_vs_implementation_correct','conclusion_justified']],
       on='case_id',validate='one_to_one')
    metric=float(f1_score(joined.human_review_operation,joined.operation_ss3,average='macro',zero_division=0))
    false_fast=int(((joined.operation_ss3=='out_of_scope') & (joined.human_review_operation!='out_of_scope')).sum())
    eligible=joined[joined.dossier_pass.isin(['True','False'])]
    dossier_rate=float((eligible.dossier_pass=='True').mean()) if len(eligible) else 0.0
    gen_eligible=generations[~generations.status.isin(['SS3_OUT_OF_SCOPE','SS3_COVERAGE'])]
    gen_rate=float((gen_eligible.status=='GENERATED_COMPLETE').mean()) if len(gen_eligible) else 0.0
    relevance=float(pd.to_numeric(joined.dossier_relevance_0_2,errors='coerce').mean())
    factual=float(pd.to_numeric(joined.narrative_factually_grounded_0_2,errors='coerce').mean())
    valid_provenance=bool(joined.norm_vs_implementation_correct.str.lower().isin(['si','sí','yes','true','no aplica']).all())
    justified=bool(joined.conclusion_justified.str.lower().isin(['si','sí','yes','true','no aplica']).all())
    gates={'operation_f1_macro_075':metric>=.75,'false_ood_le_2':false_fast<=2,
         'dossiers_090':dossier_rate>=.90,'generation_090':gen_rate>=.90,
         'substantive_relevance_mean_ge_15':relevance>=1.5,
         'factual_grounding_mean_ge_15':factual>=1.5,
         'norm_vs_implementation_no_failures':valid_provenance,
         'conclusions_justified':justified}
    verdict='APROBADO_CON_CRITERIOS' if all(gates.values()) else 'NO_APROBADO'
    result={'verdict':verdict,'n':80,'operation_f1_macro_human':metric,
       'false_ood':false_fast,'dossier_pass_rate':dossier_rate,
       'generation_complete_rate':gen_rate,'relevance_mean_0_2':relevance,
       'grounding_mean_0_2':factual,'gates':gates}
    (OUTPUT_DIR/'ss3_final_validation_verdict_v320.json').write_text(
        json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    return result

print('Dictamen configurado. Ejecutar después de las 80 generaciones y la revisión de las 80 filas.')


Dictamen configurado. Ejecutar después de las 80 generaciones y la revisión de las 80 filas.


## 15. Una sola interfaz Gradio — SS3 gobierna TODAS las preguntas

In [29]:
import gradio as gr

DEFAULT_E2E_CASE_IDS=['DOC01','COMP01','TIME01','HYP01','SAFETY01','SYSTEM01','OOD01','FOLLOW01-1','FOLLOW01-2','COMP06','TIME08','HYP09']
EXAMPLE_PROMPTS=SS3_TEST_80.question.head(12).tolist()

def ui_ask(q,history,session):
    session=session or {'previous':None,'turns':[]}
    result=ss3_run_question(q,previous=session['previous'],with_llm=True)
    session['previous']={'question':q,'provinces':result['plan']['provinces'],
      'sources':result['plan']['sources'],'periods':result['plan']['periods'],
      'retrieval_query':result['plan']['retrieval_query']}
    session['turns'].append(result)
    history=(history or [])+[{'role':'user','content':q},{'role':'assistant','content':result['answer'] or result['status']}]
    return history,session,result['plan'],result['evidence'],result['validation']

def ui_preflight():
    df,meta=ss3_evaluate_80_prellm()
    return meta,str((OUTPUT_DIR/'ss3_authority_80_plans_dossiers_v320.csv').resolve())

def ui_generations(n):
    ids=(DEFAULT_E2E_CASE_IDS + [x for x in SS3_TEST_80.case_id.tolist() if x not in DEFAULT_E2E_CASE_IDS])[:int(n)]
    frame=ss3_generate_cases(ids)
    return {'n':len(frame),'status_counts':frame.status.value_counts().to_dict()},str((OUTPUT_DIR/'ss3_ministral_e2e_v320.csv').resolve())

with gr.Blocks(title='Observatorio NOA · SS3 Planificador Único') as demo:
    gr.Markdown('## Observatorio NOA · Planificación **exclusiva SS3** → Ministral 8B')
    gr.Markdown('Sin rutas A/B, sin planner determinístico. Recuperación cerrada v3.0.3; todas las decisiones semánticas quedan auditadas.')
    with gr.Tab('Conversación y auditoría'):
        chat=gr.Chatbot(height=430)
        query=gr.Textbox(label='Pregunta',placeholder='Hacé una consulta en lenguaje natural sobre el corpus del Observatorio')
        submit=gr.Button('Analizar con SS3 + Ministral',variant='primary')
        session=gr.State({'previous':None,'turns':[]})
        with gr.Accordion('Plan decidido por SS3',open=True):audit_plan=gr.JSON(label='Clasificación, clases y vocabulario activo')
        with gr.Accordion('Dossier y validación',open=False):
            audit_evidence=gr.JSON(label='Evidencias')
            audit_validation=gr.JSON(label='Afirmaciones verificadas')
        submit.click(ui_ask,[query,chat,session],[chat,session,audit_plan,audit_evidence,audit_validation])
        query.submit(ui_ask,[query,chat,session],[chat,session,audit_plan,audit_evidence,audit_validation])
        gr.Examples(EXAMPLE_PROMPTS,inputs=query)
    with gr.Tab('Prueba externa · 80 casos'):
        btn=gr.Button('Ejecutar los 80 planes y dossiers (sin LLM)')
        metrics=gr.JSON(label='Resultados SS3')
        csvfile=gr.File(label='Descarga trazable · 80 casos')
        btn.click(ui_preflight,outputs=[metrics,csvfile])
        n=gr.Slider(1,80,value=12,step=1,label='Número de generaciones reales con Ministral')
        btn2=gr.Button('Ejecutar pruebas SS3 → Ministral en GPU')
        metrics2=gr.JSON(label='Estado de generaciones')
        genfile=gr.File(label='Descarga de respuestas')
        btn2.click(ui_generations,inputs=n,outputs=[metrics2,genfile])
    with gr.Tab('Método y criterios de validación'):
        gr.Markdown('SS3 es el único planificador. Sólo se emplean reglas de integridad para el alcance explícito de fuentes, fechas y territorios, y contratos de evidencia para la publicación. Los criterios de validación están fijados antes del test.')
        gr.JSON(value=VALIDATION_CRITERIA,label='Criterios registrados')
        gr.File(value=str((OUTPUT_DIR/'ss3_human_review_80_v320.csv').resolve()),label='Plantilla de evaluación humana')
        review_upload=gr.File(label='Adjuntar CSV de revisión humana completado',file_types=['.csv'])
        btnverdict=gr.Button('Calcular dictamen final, después de revisar las 80 respuestas')
        verdictoutput=gr.JSON(label='Veredicto científicamente condicionado')
        btnverdict.click(ss3_validation_verdict,inputs=review_upload,outputs=verdictoutput)
print('Gradio único SS3: interfaz construida. Lanzar desde la celda siguiente.')


Gradio único SS3: interfaz construida. Lanzar desde la celda siguiente.


## 16. Iniciar la aplicación en Colab

In [30]:
GRADIO_SHARE=True
if GRADIO_SHARE:
    demo.queue(default_concurrency_limit=1).launch(share=True,debug=False,show_error=True)
else:
    demo.queue(default_concurrency_limit=1).launch(share=False,debug=False,show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be6e82ed6404112396.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 17. Lectura científica de resultados

Esta versión **no hace una comparación de planificadores**: examina directamente la suficiencia de SS3 como único planificador. Los resultados de entrenamiento y el banco sintético son provisionales hasta auditoría experta. La evidencia concluyente requiere generar y revisar respuestas reales de Ministral en el corpus cerrado: las métricas de clasificación, por sí solas, no validan narrativas. No se declara éxito si no están ejecutadas las etapas 80/80 y revisadas las afirmaciones.

Criterios de trazabilidad: `ss3_only_full_audit_v320.jsonl`, `ss3_authority_80_plans_dossiers_v320.csv`, `ss3_ministral_e2e_v320.csv`, y `ss3_human_review_80_v320.csv`.

Referencia técnica de PySS3: [API del clasificador](https://pyss3.readthedocs.io/en/latest/api/index.html). El parámetro `def_cat='most-probable'` es una opción documentada; `n_grams` se pasa a `fit`, no al constructor. La confianza de SS3 no es una probabilidad calibrada.